In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con).to_string(index=False))

In [ ]:
q1 = "SELECT COUNT(*) AS rows_total, SUM(analysed) AS analysed FROM matches"
print(pd.read_sql(q1, con).to_string(index=False))

q2 = """
SELECT t.name, COUNT(*) AS matches
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1 AND m.file_year >= 2015
GROUP BY t.name
ORDER BY matches DESC
"""
print(pd.read_sql(q2, con).to_string(index=False))

In [ ]:
#matches per year and per surface
q3 = """
SELECT file_year, surface, COUNT(*) AS matches
FROM matches
WHERE file_year >= 2015
GROUP BY file_year, surface
ORDER BY file_year, surface
"""
print(pd.read_sql(q3, con).to_string(index=False))

In [ ]:
# same as above but in a table 
df3 = pd.read_sql(q3, con)
print(df3.pivot(index="file_year", columns="surface", values="matches").to_string())

In [ ]:
#percentage of service points that ended in ace
q4 = """
SELECT t.name,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name
ORDER BY aces_pct DESC
"""
print(pd.read_sql(q4, con).to_string(index=False))

In [ ]:
#same above but evolution by years
q5 = """
SELECT t.name,
       m.file_year,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name, m.file_year
HAVING COUNT(*) >= 20
ORDER BY t.name, m.file_year
"""
df5 = pd.read_sql(q5, con)

print(df5.pivot(index="name", columns="file_year", values="aces_pct").to_string())
print()
print(df5.pivot(index="name", columns="file_year", values="matches_used").to_string())

In [ ]:
con.execute("DROP VIEW IF EXISTS serve_obs")
con.execute("""
CREATE VIEW serve_obs AS
SELECT match_id, file_year, event_no, analysed, surface, best_of, indoor,
       winner_id AS server_id, loser_id AS returner_id,
       w_svpt AS svpt, w_1stWon + w_2ndWon AS sv_won, w_ace AS ace
FROM matches
WHERE retired = 0
  AND w_svpt IS NOT NULL AND w_1stWon IS NOT NULL AND w_2ndWon IS NOT NULL
  AND w_ace IS NOT NULL AND w_svpt > 0
UNION ALL
SELECT match_id, file_year, event_no, analysed, surface, best_of, indoor,
       loser_id, winner_id,
       l_svpt, l_1stWon + l_2ndWon, l_ace
FROM matches
WHERE retired = 0
  AND l_svpt IS NOT NULL AND l_1stWon IS NOT NULL AND l_2ndWon IS NOT NULL
  AND l_ace IS NOT NULL AND l_svpt > 0
""")
con.commit()

In [ ]:
print(pd.read_sql("""
SELECT COUNT(*) AS rows_total,
       SUM(analysed) AS analysed_rows,
       SUM(CASE WHEN sv_won > svpt OR ace > svpt OR ace > sv_won THEN 1 ELSE 0 END) AS impossible_rows
FROM serve_obs
""", con).to_string(index=False))

print(pd.read_sql("""
SELECT surface,
       COUNT(*) AS rows_n,
       ROUND(100.0 * SUM(sv_won) / SUM(svpt), 2) AS service_pts_won_pct,
       ROUND(100.0 * SUM(ace) / SUM(svpt), 2) AS aces_pct
FROM serve_obs
WHERE file_year >= 2015
GROUP BY surface
ORDER BY service_pts_won_pct DESC
""", con).to_string(index=False))

In [ ]:
import numpy as np

obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)
tournament_names = pd.read_sql("SELECT event_no, name FROM tournaments", con)
print(obs.shape)


def window_sums(df, keys, prefix):
    base = df.groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    parts = [base.assign(file_year=base["file_year"] + d) for d in (-1, 0, 1)]
    out = pd.concat(parts).groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    return out.rename(columns={"svpt": prefix + "_svpt", "val": prefix + "_val"})


def role_excess(t, d, pid_col, sf, prefix, mu, k):
    dd = d.assign(pid=d[pid_col])
    tt = t.assign(pid=t[pid_col])
    w = window_sums(dd, sf + ["pid"], prefix + "w")
    e = window_sums(dd, sf + ["pid", "event_no"], prefix + "e")
    tt = tt.merge(w, on=sf + ["pid", "file_year"], how="left")
    tt = tt.merge(e, on=sf + ["pid", "event_no", "file_year"], how="left")
    n = (tt[prefix + "w_svpt"].fillna(0) - tt[prefix + "e_svpt"].fillna(0)).to_numpy()
    v = (tt[prefix + "w_val"].fillna(0) - tt[prefix + "e_val"].fillna(0)).to_numpy()
    return (v - mu * n) / (n + k), n


def expected_values(obs, value, by_surface=True, k=500, analysed_only=True):
    sf = ["surface"] if by_surface else []
    d = obs.rename(columns={value: "val"})[["match_id", "file_year", "event_no", "analysed", "surface",
                                              "server_id", "returner_id", "svpt", "val"]]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)

    t = t.merge(window_sums(d, sf, "wt"), on=sf + ["file_year"], how="left")
    t = t.merge(window_sums(d, sf + ["event_no"], "et"), on=sf + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()

    ex_s, n_s = role_excess(t, d, "server_id", sf, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", sf, "r", mu, k)

    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t

In [ ]:
res_spw = expected_values(obs, "sv_won")
obs_pct = 100 * res_spw["val"].sum() / res_spw["svpt"].sum()
exp_pct = 100 * (res_spw["exp"] * res_spw["svpt"]).sum() / res_spw["svpt"].sum()
print(len(res_spw), "rows | observed", round(float(obs_pct), 2), "% | expected", round(float(exp_pct), 2), "%")
print("rows with no server baseline:", int((res_spw["n_srv"] == 0).sum()),
      "| no returner baseline:", int((res_spw["n_ret"] == 0).sum()))

In [ ]:
def speed_table(res, B=1000, min_matches=20, seed=1):
    m = res.groupby(["match_id", "event_no"], as_index=False).agg(
        rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
    m = m[m["rows"] == 2]
    rng = np.random.default_rng(seed)
    out = []
    for ev, g in m.groupby("event_no"):
        if len(g) < min_matches:
            continue
        s, r = g["svpt"].to_numpy(), g["resid"].to_numpy()
        idx = rng.integers(0, len(g), size=(B, len(g)))
        boots = 100 * r[idx].sum(axis=1) / s[idx].sum(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"event_no": ev, "matches": len(g),
                    "speed_pts": round(100 * r.sum() / s.sum(), 2),
                    "ci_low": round(float(lo), 2), "ci_high": round(float(hi), 2)})
    return (pd.DataFrame(out).merge(tournament_names, on="event_no")
            .sort_values("speed_pts", ascending=False)[["name", "matches", "speed_pts", "ci_low", "ci_high"]])

print("Service points won, relative to other events on the same surface")
print(speed_table(res_spw).to_string(index=False))

In [ ]:
res_ace = expected_values(obs, "ace")
print("Aces, relative to other events on the same surface")
print(speed_table(res_ace).to_string(index=False))
print()
res_spw_all = expected_values(obs, "sv_won", by_surface=False)
print("Service points won, relative to ALL other events (surfaces mixed)")
print(speed_table(res_spw_all).to_string(index=False))

In [ ]:
res_all = expected_values(obs, "sv_won", analysed_only=False)
o = 100 * res_all["val"].sum() / res_all["svpt"].sum()
e = 100 * (res_all["exp"] * res_all["svpt"]).sum() / res_all["svpt"].sum()
print(len(res_all), "rows | observed", round(float(o), 2), "% | expected", round(float(e), 2), "%")

In [ ]:
t3 = speed_table(res_spw_all)
print(t3.iloc[:7].to_string(index=False))
print(t3.iloc[7:].to_string(index=False))

In [ ]:
obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface, indoor,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)
print(obs.shape, obs["indoor"].value_counts(dropna=False).to_dict())

In [ ]:
def expected_values_k(obs, value, keys, k=500, analysed_only=True):
    base_cols = ["match_id", "file_year", "event_no", "analysed", "server_id", "returner_id", "svpt"]
    d = obs.rename(columns={value: "val"})[list(dict.fromkeys(base_cols + keys + ["val"]))]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)

    t = t.merge(window_sums(d, keys, "wt"), on=keys + ["file_year"], how="left")
    t = t.merge(window_sums(d, keys + ["event_no"], "et"), on=keys + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()

    ex_s, n_s = role_excess(t, d, "server_id", keys, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", keys, "r", mu, k)

    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t

In [ ]:
res_roof = expected_values_k(obs, "sv_won", ["surface", "indoor"])
print(len(res_roof), "rows | observed",
      round(float(100 * res_roof["val"].sum() / res_roof["svpt"].sum()), 2), "% | expected",
      round(float((100 * res_roof["exp"] * res_roof["svpt"]).sum() / res_roof["svpt"].sum()), 2), "%")
print("Service points won, relative to events with the same surface and roof")
print(speed_table(res_roof).to_string(index=False))

In [ ]:
cols = []
for k in [100, 250, 500, 1000, 2000]:
    r_ = expected_values_k(obs, "sv_won", ["surface", "indoor"], k=k)
    cols.append(speed_table(r_, B=100).set_index("name")["speed_pts"].rename(f"k={k}"))
print(pd.concat(cols, axis=1).round(2).to_string())

In [ ]:
def speed_by_year(res, B=500, min_matches=20, seed=1):
    m = res.groupby(["match_id", "event_no", "file_year"], as_index=False).agg(
        rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
    m = m[m["rows"] == 2]
    rng = np.random.default_rng(seed)
    out = []
    for (ev, yr), g in m.groupby(["event_no", "file_year"]):
        if len(g) < min_matches:
            continue
        s, r = g["svpt"].to_numpy(), g["resid"].to_numpy()
        idx = rng.integers(0, len(g), size=(B, len(g)))
        boots = 100 * r[idx].sum(axis=1) / s[idx].sum(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"event_no": ev, "year": yr, "matches": len(g),
                    "speed_pts": 100 * r.sum() / s.sum(), "half_width": (hi - lo) / 2})
    return pd.DataFrame(out).merge(tournament_names, on="event_no")

by_year = speed_by_year(res_roof)
print(by_year.pivot(index="name", columns="year", values="speed_pts").round(1).to_string())
print()
print(by_year.pivot(index="name", columns="year", values="half_width").round(1).to_string())

In [ ]:
by_year = speed_by_year(res_roof)
print(by_year.pivot(index="name", columns="year", values="speed_pts").round(1).to_string())
print()
print(by_year.pivot(index="name", columns="year", values="half_width").round(1).to_string())

In [ ]:
for name in ["pd", "np", "con", "obs", "expected_values_k", "speed_table", "tournament_names", "res_roof", "by_year"]:
    print(name, name in globals())

In [ ]:
meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Service points won above expectation (percentage points). Baseline: other events with the same "
               "surface and roof, same players, +/-1 year, shrinkage k=500. Interval: bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}

tbl_year = by_year.rename(columns={"year": "file_year", "speed_pts": "serve_excess_pp"})[
    ["event_no", "name", "file_year", "matches", "serve_excess_pp", "half_width"]].copy()
for k_, v_ in meta.items():
    tbl_year[k_] = v_
tbl_year.to_sql("serve_advantage_by_year", con, if_exists="replace", index=False)

pooled = speed_table(res_roof).rename(columns={"speed_pts": "serve_excess_pp"})
for k_, v_ in meta.items():
    pooled[k_] = v_
pooled.to_sql("serve_advantage_pooled", con, if_exists="replace", index=False)
con.commit()

print(pd.read_sql("SELECT COUNT(*) AS by_year_rows FROM serve_advantage_by_year", con).to_string(index=False))
print(pd.read_sql("SELECT COUNT(*) AS pooled_rows FROM serve_advantage_pooled", con).to_string(index=False))

In [ ]:
for name in ["pd", "np", "con"]:
    print(name, name in globals())
print(pd.read_sql("SELECT COUNT(*) AS n FROM serve_obs", con).iloc[0, 0], "rows in serve_obs")

In [ ]:
def season_card(year, min_matches=30, top=15, sort_by="avg_elo"):
    p = {"y": year, "mm": min_matches}
    base = pd.read_sql("""
        SELECT e.player_id,
               COUNT(*) AS matches,
               SUM(e.result) AS wins,
               ROUND(AVG(e.elo_before)) AS avg_elo,
               ROUND(MAX(e.elo_after)) AS peak_elo,
               ROUND(AVG(o.elo_before)) AS avg_opp_elo,
               SUM(CASE WHEN e.expected < 0.5 THEN 1 ELSE 0 END) AS matches_as_underdog,
               SUM(CASE WHEN e.expected < 0.5 AND e.result = 1 THEN 1 ELSE 0 END) AS wins_as_underdog,
               ROUND(SUM(e.result - e.expected), 1) AS wins_above_expected
        FROM elo_long e
        JOIN elo_long o ON o.match_id = e.match_id AND o.player_id = e.opponent_id
        JOIN matches m ON m.match_id = e.match_id
        WHERE m.file_year = :y
        GROUP BY e.player_id
        HAVING COUNT(*) >= :mm
    """, con, params=p)

    titles = pd.read_sql("SELECT winner_id AS player_id, COUNT(*) AS titles FROM matches "
                         "WHERE round = 'F' AND file_year = :y GROUP BY winner_id", con, params={"y": year})
    lost = pd.read_sql("SELECT loser_id AS player_id, COUNT(*) AS finals_lost FROM matches "
                       "WHERE round = 'F' AND file_year = :y GROUP BY loser_id", con, params={"y": year})
    srv = pd.read_sql("SELECT server_id AS player_id, "
                      "ROUND(100.0 * SUM(sv_won) / SUM(svpt), 1) AS svc_pts_won_pct, "
                      "ROUND(100.0 * SUM(ace) / SUM(svpt), 1) AS aces_pct "
                      "FROM serve_obs WHERE file_year = :y GROUP BY server_id", con, params={"y": year})
    ret = pd.read_sql("SELECT returner_id AS player_id, "
                      "ROUND(100.0 * SUM(svpt - sv_won) / SUM(svpt), 1) AS ret_pts_won_pct "
                      "FROM serve_obs WHERE file_year = :y GROUP BY returner_id", con, params={"y": year})
    names = pd.read_sql("SELECT player_id, name FROM players", con)

    d = base
    for t in (titles, lost, srv, ret, names):
        d = d.merge(t, on="player_id", how="left")
    d[["titles", "finals_lost"]] = d[["titles", "finals_lost"]].fillna(0).astype(int)

    z = 1.96
    n, pw = d["matches"], d["wins"] / d["matches"]
    center = (pw + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(pw * (1 - pw) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    d["win_pct"] = (100 * pw).round(1)
    d["win_pct_low"] = (100 * (center - half)).round(1)
    d["win_pct_high"] = (100 * (center + half)).round(1)

    cols = ["name", "matches", "wins", "win_pct", "win_pct_low", "win_pct_high", "titles", "finals_lost",
            "avg_elo", "peak_elo", "avg_opp_elo", "wins_as_underdog", "matches_as_underdog",
            "wins_above_expected", "svc_pts_won_pct", "ret_pts_won_pct", "aces_pct"]
    return d.sort_values(sort_by, ascending=False).head(top)[cols]

In [ ]:
print(season_card(2024, top=3).to_string(index=False))

In [ ]:
frames = []
for yr in range(2015, 2027):
    mm = 25 if yr == 2026 else 30
    c = season_card(yr, min_matches=mm, top=40).copy()
    c.insert(0, "season", yr)
    c["min_matches"] = mm
    c["as_of"] = pd.read_sql("SELECT MAX(tourney_start) AS d FROM matches WHERE file_year = :y",
                             con, params={"y": yr}).iloc[0, 0]
    frames.append(c)

cards = pd.concat(frames, ignore_index=True)
cards["source"] = "Calculated from TennisMyLife match data"
cards["licence"] = "Derived aggregate; non-commercial, see data/Sources.md"
cards["method"] = ("Per player and season (TML file year), ATP tour-level main-draw matches in our data. Elo v2 ratings; "
                   "wins above expected = sum of (result - expected probability); win % interval = Wilson 95%.")
cards["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
cards.to_sql("player_season_cards", con, if_exists="replace", index=False)
con.commit()

print(cards.groupby("season").size().to_string())

In [ ]:
import numpy as np

con.execute("DROP VIEW IF EXISTS server_rows")
con.execute("""
CREATE VIEW server_rows AS
SELECT match_id, file_year, event_no, surface, indoor, winner_id AS server_id,
       w_svpt AS svpt, w_ace AS ace, w_df AS df, w_1stIn AS first_in, w_1stWon AS first_won,
       w_2ndWon AS second_won, w_SvGms AS sv_games, w_bpFaced AS bp_faced, w_bpSaved AS bp_saved
FROM matches
WHERE retired = 0 AND w_svpt > 0 AND w_ace IS NOT NULL AND w_df IS NOT NULL AND w_1stIn IS NOT NULL
  AND w_1stWon IS NOT NULL AND w_2ndWon IS NOT NULL AND w_SvGms IS NOT NULL
  AND w_bpFaced IS NOT NULL AND w_bpSaved IS NOT NULL
UNION ALL
SELECT match_id, file_year, event_no, surface, indoor, loser_id,
       l_svpt, l_ace, l_df, l_1stIn, l_1stWon, l_2ndWon, l_SvGms, l_bpFaced, l_bpSaved
FROM matches
WHERE retired = 0 AND l_svpt > 0 AND l_ace IS NOT NULL AND l_df IS NOT NULL AND l_1stIn IS NOT NULL
  AND l_1stWon IS NOT NULL AND l_2ndWon IS NOT NULL AND l_SvGms IS NOT NULL
  AND l_bpFaced IS NOT NULL AND l_bpSaved IS NOT NULL
""")
con.commit()

sr = pd.read_sql("SELECT * FROM server_rows WHERE file_year >= 2015", con)
sr["spw_num"] = sr["first_won"] + sr["second_won"]
sr["second_in_den"] = sr["svpt"] - sr["first_in"]
sr["hold_num"] = sr["sv_games"] - (sr["bp_faced"] - sr["bp_saved"])
ok = sr["hold_num"] >= 0
sr["hold_ok"] = np.where(ok, sr["hold_num"], 0)
sr["games_ok"] = np.where(ok, sr["sv_games"], 0)
print(len(sr), "server rows | negative holds:", int((~ok).sum()),
      "| second-serve denominators <= 0:", int((sr["second_in_den"] <= 0).sum()))
print("columns present:", "hold_ok" in sr.columns, "games_ok" in sr.columns)

In [ ]:
info = pd.read_sql("SELECT match_id, best_of, minutes, score FROM matches WHERE file_year >= 2015", con)
num_cols = ["spw_num", "ace", "df", "first_in", "first_won", "second_won", "hold_num", "svpt", "sv_games","second_in_den", "hold_ok", "games_ok"]
pm = sr.groupby(["match_id", "file_year", "event_no", "surface", "indoor"], as_index=False)[num_cols].sum()
pm = pm.merge(info, on="match_id", how="left")
pm["is_paris"] = pm["event_no"].astype(str) == "352"
bo3 = pm["best_of"] == 3
pm["min_den"] = (bo3 & pm["minutes"].notna()).astype(int)
pm["min_num"] = np.where(pm["min_den"] == 1, pm["minutes"], 0.0)
pm["tb_den"] = bo3.astype(int)
pm["tb_num"] = (bo3 & pm["score"].str.contains(r"7-6|6-7", na=False)).astype(int)

metrics = {
    "service_pts_won_pct": ("spw_num", "svpt", 100),
    "aces_pct": ("ace", "svpt", 100),
    "double_faults_pct": ("df", "svpt", 100),
    "first_serve_in_pct": ("first_in", "svpt", 100),
    "first_serve_pts_won_pct": ("first_won", "first_in", 100),
    "second_serve_pts_won_pct": ("second_won", "second_in_den", 100),
    "service_games_held_pct": ("hold_ok", "games_ok", 100),
    "minutes_per_match_bo3": ("min_num", "min_den", 1),
    "matches_with_tiebreak_pct_bo3": ("tb_num", "tb_den", 100),
}


def boot_ratio(num, den, scale, rng, B=300):
    idx = rng.integers(0, len(num), size=(B, len(num)))
    return np.percentile(scale * num[idx].sum(axis=1) / den[idx].sum(axis=1), [2.5, 97.5])


latest = pd.read_sql("SELECT MAX(tourney_start) AS d FROM matches", con).iloc[0, 0]
ps = pd.read_sql("SELECT file_year, MIN(tourney_start) AS s FROM matches WHERE event_no = '352' GROUP BY file_year", con)
partial_years = set(ps[pd.to_datetime(ps["s"]) >= pd.Timestamp(latest) - pd.Timedelta(days=6)]["file_year"])

rng = np.random.default_rng(1)
rows = []
for yr in [y for y in sorted(pm["file_year"].unique()) if y <= 2025]:
    py = pm[pm["file_year"] == yr]
    groups = {
        "Paris Masters": py["is_paris"],
        "Rest of the tour": ~py["is_paris"],
        "Other indoor hard events": (~py["is_paris"]) & (py["indoor"] == "I") & (py["surface"] == "Hard"),
    }
    for gname, mask in groups.items():
        g = py[mask]
        if len(g) < (10 if gname == "Paris Masters" else 30):
            continue
        for metric, (nc, dc, scale) in metrics.items():
            num, den = g[nc].to_numpy(float), g[dc].to_numpy(float)
            if den.sum() == 0:
                continue
            lo, hi = boot_ratio(num, den, scale, rng)
            rows.append({"metric": metric, "group": gname, "file_year": int(yr),
                         "value": scale * num.sum() / den.sum(), "ci_low": float(lo), "ci_high": float(hi),
                         "matches": len(g), "partial": int(gname == "Paris Masters" and yr in partial_years)})

by_year_tbl = pd.DataFrame(rows)
print("latest tournament start in the data:", latest, "| partial Paris years:", sorted(partial_years))
print(by_year_tbl[by_year_tbl["metric"] == "service_pts_won_pct"]
      .pivot(index="file_year", columns="group", values="value").round(2).to_string())

In [ ]:
def pooled_difference(pm, years, B=300, seed=2):
    rng = np.random.default_rng(seed)
    sel = pm[pm["file_year"].isin(years)]
    p, r = sel[sel["is_paris"]], sel[~sel["is_paris"]]
    P = {m: (p[nc].to_numpy(float), p[dc].to_numpy(float)) for m, (nc, dc, sc) in metrics.items()}
    R = {m: (r[nc].to_numpy(float), r[dc].to_numpy(float)) for m, (nc, dc, sc) in metrics.items()}
    diffs = {m: [] for m in metrics}
    for _ in range(B):
        ip, ir = rng.integers(0, len(p), len(p)), rng.integers(0, len(r), len(r))
        for m, (nc, dc, sc) in metrics.items():
            diffs[m].append(sc * P[m][0][ip].sum() / P[m][1][ip].sum() - sc * R[m][0][ir].sum() / R[m][1][ir].sum())
    out = []
    for m, (nc, dc, sc) in metrics.items():
        pv, rv = sc * P[m][0].sum() / P[m][1].sum(), sc * R[m][0].sum() / R[m][1].sum()
        lo, hi = np.percentile(diffs[m], [2.5, 97.5])
        out.append({"metric": m, "paris": pv, "rest_of_tour": rv, "difference": pv - rv, "ci_low": float(lo),
                    "ci_high": float(hi), "unit": "minutes" if sc == 1 else "percentage points",
                    "paris_matches": len(p), "rest_matches": len(r),
                    "years": f"{min(years)}-{max(years)}"})
    return pd.DataFrame(out)


pooled_tbl = pooled_difference(pm, list(range(2015, 2026)))
print(pooled_tbl[["metric", "paris", "rest_of_tour", "difference", "ci_low", "ci_high"]].round(2).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Per statistic: sums over servers' rows (retired matches excluded). Bands: 95% bootstrap over matches. "
               "Rest of the tour = all other tour-level events that year. Minutes and tiebreaks: best-of-3 matches only. "
               "Service games held derived from break points faced and saved."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    by_year_tbl[k_] = v_
    pooled_tbl[k_] = v_
by_year_tbl.to_sql("paris_characteristics_by_year", con, if_exists="replace", index=False)
pooled_tbl.to_sql("paris_vs_tour_pooled", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(by_year_tbl), "yearly rows,", len(pooled_tbl), "pooled rows")

In [ ]:
bad = sr[sr["hold_num"] < 0]
print(bad.groupby("file_year").size().to_string())
print("in Paris:", int((bad["event_no"].astype(str) == "352").sum()))

In [ ]:
for yrs, label in [(list(range(2015, 2026)), "2015-2025"),
                   ([y for y in range(2015, 2026) if y not in (2017, 2018)], "without 2017-2018")]:
    t = pooled_difference(pm, yrs)
    print(label)
    print(t[t["metric"] == "service_games_held_pct"][["paris", "rest_of_tour", "difference", "ci_low", "ci_high"]].round(2).to_string(index=False))

In [ ]:
print(pd.read_sql("""
    SELECT metric, ROUND(paris, 2) AS paris, ROUND(rest_of_tour, 2) AS rest_of_tour, ROUND(difference, 2) AS difference
    FROM paris_vs_tour_pooled WHERE metric = 'service_games_held_pct'
""", con).to_string(index=False))

In [ ]:
tm = pd.read_sql("SELECT event_no, name FROM tournaments WHERE level = 'M'", con)
tm["event_no"] = tm["event_no"].astype(str)
pmm = pm.assign(event_no=pm["event_no"].astype(str)).merge(tm, on="event_no")
pmm = pmm[pmm["file_year"] <= 2025]

rng = np.random.default_rng(3)
rows = []


def add_row(g, label, yr):
    num, den = g["hold_ok"].to_numpy(float), g["games_ok"].to_numpy(float)
    if len(g) < 20 or den.sum() == 0:
        return
    lo, hi = boot_ratio(num, den, 100, rng)
    rows.append({"name": label, "file_year": int(yr), "matches": len(g),
                 "hold_pct": 100 * num.sum() / den.sum(), "ci_low": float(lo), "ci_high": float(hi)})


for (nm, yr), g in pmm.groupby(["name", "file_year"]):
    add_row(g, nm, yr)
for yr, g in pmm.groupby("file_year"):
    add_row(g, "All Masters 1000 (pooled)", yr)

hold_tbl = pd.DataFrame(rows)
print(hold_tbl.pivot(index="file_year", columns="name", values="hold_pct").round(1).to_string())

meta2 = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Service games held = (service games - (break points faced - break points saved)) / service games, "
               "summed over matches; rows with impossible counts excluded; retired matches excluded. "
               "Bars: 95% bootstrap over matches, at least 20 matches per tournament-year."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta2.items():
    hold_tbl[k_] = v_
hold_tbl.to_sql("masters_service_games_held", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(hold_tbl), "rows")

In [ ]:
def boot_ratio(num, den, scale, rng, B=300):
    idx = rng.integers(0, len(num), size=(B, len(num)))
    return np.percentile(scale * num[idx].sum(axis=1) / den[idx].sum(axis=1), [2.5, 97.5])


valid = (sr["hold_num"] >= 0) & (sr["bp_saved"] <= sr["bp_faced"])
sr["bp_faced_ok"] = np.where(valid, sr["bp_faced"], 0)
sr["bp_conv_ok"] = np.where(valid, sr["bp_faced"] - sr["bp_saved"], 0)
sr["games_bp_ok"] = np.where(valid, sr["sv_games"], 0)
pm_bp = sr.groupby(["match_id", "file_year", "event_no", "surface", "indoor"], as_index=False)[
    ["bp_faced_ok", "bp_conv_ok", "games_bp_ok"]].sum()
pm_bp["is_paris"] = pm_bp["event_no"].astype(str) == "352"
pm_bp = pm_bp[pm_bp["file_year"] <= 2025]

fav = pd.read_sql("""
    SELECT m.match_id, m.file_year, m.event_no, m.surface, m.indoor, e.expected AS p_winner
    FROM elo_long e JOIN matches m ON m.match_id = e.match_id
    WHERE e.result = 1 AND m.file_year BETWEEN 2015 AND 2025
""", con)
fav = fav[fav["p_winner"] != 0.5].copy()
fav["p_fav"] = np.maximum(fav["p_winner"], 1 - fav["p_winner"])
fav["fav_won"] = (fav["p_winner"] > 0.5).astype(float)
fav["fav_gap"] = fav["fav_won"] - fav["p_fav"]
fav["is_paris"] = fav["event_no"].astype(str) == "352"
fav["one"] = 1.0
for nm, lo_, hi_ in [("b1", 0.5, 0.6), ("b2", 0.6, 0.75), ("b3", 0.75, 1.01)]:
    inb = (fav["p_fav"] > lo_) & (fav["p_fav"] <= hi_)
    fav["n_" + nm] = inb.astype(float)
    fav["w_" + nm] = (inb & (fav["fav_won"] == 1)).astype(float)

bp_mets = {
    "break_points_converted_pct": ("bp_conv_ok", "bp_faced_ok", 100),
    "break_points_per_100_service_games": ("bp_faced_ok", "games_bp_ok", 100),
}
fav_mets = {
    "favourite_win_pct": ("fav_won", "one", 100),
    "avg_favourite_probability_pct": ("p_fav", "one", 100),
    "favourite_wins_above_expected_pct": ("fav_gap", "one", 100),
    "favourite_win_pct_50_60": ("w_b1", "n_b1", 100),
    "favourite_win_pct_60_75": ("w_b2", "n_b2", 100),
    "favourite_win_pct_75_plus": ("w_b3", "n_b3", 100),
    "matches_with_75_plus_favourite_pct": ("n_b3", "one", 100),
}
print("rows with more break points saved than faced:", int((sr["bp_saved"] > sr["bp_faced"]).sum()))
print("matches with break-point data:", len(pm_bp), "| Paris:", int(pm_bp["is_paris"].sum()))
print("matches in the favourite table:", len(fav), "| Paris:", int(fav["is_paris"].sum()))

In [ ]:
def yearly_series(df, mets, years, seed=7, B=300, min_paris=10, min_other=30):
    rng = np.random.default_rng(seed)
    rows = []
    for yr in years:
        dy = df[df["file_year"] == yr]
        groups = {"Paris Masters": dy["is_paris"],
                  "Rest of the tour": ~dy["is_paris"],
                  "Other indoor hard events": (~dy["is_paris"]) & (dy["indoor"] == "I") & (dy["surface"] == "Hard")}
        for gname, mask in groups.items():
            g = dy[mask]
            if len(g) < (min_paris if gname == "Paris Masters" else min_other):
                continue
            for metric, (nc, dc, sc) in mets.items():
                num, den = g[nc].to_numpy(float), g[dc].to_numpy(float)
                if den.sum() == 0:
                    continue
                lo, hi = boot_ratio(num, den, sc, rng, B)
                rows.append({"metric": metric, "group": gname, "file_year": int(yr), "value": sc * num.sum() / den.sum(),
                             "ci_low": float(lo), "ci_high": float(hi), "matches": len(g)})
    return pd.DataFrame(rows)


def pooled_vs(df, mets, years, ref, B=300, seed=5):
    rng = np.random.default_rng(seed)
    sel = df[df["file_year"].isin(years)]
    p = sel[sel["is_paris"]]
    rest = sel[~sel["is_paris"]]
    r = rest if ref == "Rest of the tour" else rest[(rest["indoor"] == "I") & (rest["surface"] == "Hard")]
    P = {m: (p[a].to_numpy(float), p[b].to_numpy(float)) for m, (a, b, s) in mets.items()}
    R = {m: (r[a].to_numpy(float), r[b].to_numpy(float)) for m, (a, b, s) in mets.items()}
    diffs = {m: [] for m in mets}
    for _ in range(B):
        ip, ir = rng.integers(0, len(p), len(p)), rng.integers(0, len(r), len(r))
        for m, (a, b, s) in mets.items():
            diffs[m].append(s * P[m][0][ip].sum() / P[m][1][ip].sum() - s * R[m][0][ir].sum() / R[m][1][ir].sum())
    out = []
    for m, (a, b, s) in mets.items():
        pv, rv = s * P[m][0].sum() / P[m][1].sum(), s * R[m][0].sum() / R[m][1].sum()
        lo, hi = np.percentile(diffs[m], [2.5, 97.5])
        out.append({"metric": m, "reference": ref, "paris": pv, "reference_value": rv, "difference": pv - rv,
                    "ci_low": float(lo), "ci_high": float(hi),
                    "unit": "break points per 100 service games" if m == "break_points_per_100_service_games" else "percentage points",
                    "paris_matches": len(p), "reference_matches": len(r), "years": f"{min(years)}-{max(years)}"})
    return pd.DataFrame(out)


years = list(range(2015, 2026))
yearly = pd.concat([yearly_series(pm_bp, bp_mets, years), yearly_series(fav, fav_mets, years)], ignore_index=True)
pooled = pd.concat([pooled_vs(pm_bp, bp_mets, years, "Rest of the tour"),
                    pooled_vs(fav, fav_mets, years, "Rest of the tour"),
                    pooled_vs(pm_bp, bp_mets, years, "Other indoor hard events"),
                    pooled_vs(fav, fav_mets, years, "Other indoor hard events")], ignore_index=True)

cols = ["metric", "reference", "paris", "reference_value", "difference", "ci_low", "ci_high"]
print(pooled[cols].round(2).to_string(index=False))

In [ ]:
meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Break points: sums over servers' rows, retired matches and rows with impossible counts excluded. "
               "Favourite = the player rated higher by our Elo v2 (calibrated probability before the match); retirements count. "
               "Bands: 95% bootstrap over matches. Completed editions 2015-2025."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    yearly[k_] = v_
    pooled[k_] = v_
yearly.to_sql("paris_extra_by_year", con, if_exists="replace", index=False)
pooled.to_sql("paris_extra_pooled", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(yearly), "yearly rows,", len(pooled), "pooled rows")

In [ ]:
t14 = pd.read_sql("SELECT event_no, name, indoor_outdoor FROM tournaments WHERE surface = 'Hard'", con)
t14["event_no"] = t14["event_no"].astype(str)
pmh = pm.assign(event_no=pm["event_no"].astype(str)).merge(t14, on="event_no")
pmh = pmh[pmh["file_year"] <= 2025]
print(pmh.groupby("name").size().sort_values(ascending=False).to_string())

B = 500
rng = np.random.default_rng(11)


def boot_dist(n, d):
    idx = rng.integers(0, len(n), size=(B, len(n)))
    return 100 * n[idx].sum(axis=1) / d[idx].sum(axis=1)


arr = {nm: (g["ace"].to_numpy(float), g["svpt"].to_numpy(float)) for nm, g in pmh.groupby("name")}
others = pmh[pmh["event_no"] != "352"]
OTH = "Other hard-court events (pooled)"
arr[OTH] = (others["ace"].to_numpy(float), others["svpt"].to_numpy(float))
dist = {nm: boot_dist(n, d) for nm, (n, d) in arr.items()}
roof = dict(zip(t14["name"], t14["indoor_outdoor"]))
cnt = pmh.groupby("name").size().to_dict()
cnt[OTH] = len(others)
paris_val = 100 * arr["Paris Masters"][0].sum() / arr["Paris Masters"][1].sum()

rows = []
for nm, (n, d) in arr.items():
    val = 100 * n.sum() / d.sum()
    lo, hi = np.percentile(dist[nm] - dist["Paris Masters"], [2.5, 97.5])
    rows.append({"name": nm, "matches": cnt[nm], "aces_pct": val, "diff_vs_paris": val - paris_val,
                 "ci_low": float(lo), "ci_high": float(hi), "indoor_outdoor": roof.get(nm, "")})
pooled_hard = pd.DataFrame(rows).sort_values("aces_pct", ascending=False)
print(pooled_hard.round(2).to_string(index=False))

yr_rows = []
for yr in range(2015, 2026):
    py = pmh[pmh["file_year"] == yr]
    for label, mask in [("Paris Masters", py["event_no"] == "352"),
                        ("Other hard-court events", py["event_no"] != "352")]:
        g = py[mask]
        if len(g) < 10:
            continue
        yr_rows.append({"group": label, "file_year": yr,
                        "aces_pct": 100 * g["ace"].sum() / g["svpt"].sum(), "matches": len(g)})
by_year_hard = pd.DataFrame(yr_rows)

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Aces as a share of service points, summed over servers' rows (retired matches and rows with missing "
               "statistics excluded). Hard-court events among the 14 analysed events, completed editions 2015-2025. "
               "diff_vs_paris: event minus Paris, 95% bootstrap interval over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    pooled_hard[k_] = v_
    by_year_hard[k_] = v_
pooled_hard.to_sql("aces_hard_events_pooled", con, if_exists="replace", index=False)
by_year_hard.to_sql("aces_paris_vs_hard_by_year", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(pooled_hard), "pooled rows,", len(by_year_hard), "yearly rows")

In [ ]:
print(pd.read_sql("""SELECT (SELECT COUNT(*) FROM matches) AS matches,
                            (SELECT COUNT(*) FROM elo_long)/2 AS elo_matches""", con).to_string(index=False))

In [ ]:
print(pd.read_sql("""
    SELECT tourney_start, tourney_name, tourney_level, COUNT(*) AS matches
    FROM matches WHERE file_year = 2026 AND tourney_start >= '2026-10-12'
    GROUP BY tourney_start, tourney_name, tourney_level ORDER BY tourney_start
""", con).to_string(index=False))

In [ ]:
PLAYER = "Novak Djokovic"
FIRST, LAST = 2007, 2025

ids = pd.read_sql("SELECT player_id, name FROM players WHERE name LIKE :n", con,
                  params={"n": f"%{PLAYER.split()[-1]}%"})
print(ids.to_string(index=False))
pid = ids.loc[ids["name"] == PLAYER, "player_id"].iloc[0]

hard = pd.read_sql("SELECT event_no, name, indoor_outdoor FROM tournaments WHERE surface = 'Hard'", con)
hard["event_no"] = hard["event_no"].astype(str)

d = pd.read_sql("""
    SELECT e.match_id, m.file_year, m.event_no, m.round, m.round_order, e.result, e.expected
    FROM elo_long e JOIN matches m ON m.match_id = e.match_id
    WHERE e.player_id = :p AND m.file_year BETWEEN :a AND :b
""", con, params={"p": pid, "a": FIRST, "b": LAST})
d["event_no"] = d["event_no"].astype(str)
d = d.merge(hard, on="event_no")
d["wae"] = d["result"] - d["expected"]
d["is_paris"] = d["event_no"] == "352"
print(len(d), "matches at hard-court events |", int(d["is_paris"].sum()), "at Paris")

rng = np.random.default_rng(21)
B = 2000


def boot_mean(x):
    idx = rng.integers(0, len(x), size=(B, len(x)))
    return x[idx].mean(axis=1)


rows, dist = [], {}
for nm, g in d.groupby("name"):
    x = g["wae"].to_numpy(float)
    dist[nm] = boot_mean(x)
    lo, hi = np.percentile(dist[nm], [2.5, 97.5])
    rows.append({"player": PLAYER, "name": nm, "matches": len(g), "wins": int(g["result"].sum()),
                 "win_pct": 100 * g["result"].mean(), "wae_per_100": 100 * x.mean(),
                 "ci_low": 100 * lo, "ci_high": 100 * hi, "indoor_outdoor": g["indoor_outdoor"].iloc[0],
                 "diff_paris_minus_others": np.nan, "diff_ci_low": np.nan, "diff_ci_high": np.nan})

oth = d[~d["is_paris"]]
xo = oth["wae"].to_numpy(float)
do = boot_mean(xo)
lo, hi = np.percentile(do, [2.5, 97.5])
dlo, dhi = np.percentile(dist["Paris Masters"] - do, [2.5, 97.5])
paris_mean = d[d["is_paris"]]["wae"].mean()
rows.append({"player": PLAYER, "name": "Other hard-court events (pooled)", "matches": len(oth),
             "wins": int(oth["result"].sum()), "win_pct": 100 * oth["result"].mean(),
             "wae_per_100": 100 * xo.mean(), "ci_low": 100 * lo, "ci_high": 100 * hi, "indoor_outdoor": "",
             "diff_paris_minus_others": 100 * (paris_mean - xo.mean()),
             "diff_ci_low": 100 * dlo, "diff_ci_high": 100 * dhi})
events_tbl = pd.DataFrame(rows).sort_values("wae_per_100", ascending=False)
print(events_tbl.round(1).drop(columns=["player"]).to_string(index=False))

In [ ]:
STAGE = {"R128": 0, "R64": 1, "R32": 2, "R16": 3, "QF": 4, "SF": 5, "F": 6}
par = d[d["is_paris"]]
yr_rows = []
for yr, g in par.groupby("file_year"):
    last = g.sort_values("round_order").iloc[-1]
    if last["round"] == "F" and last["result"] == 1:
        res, stage = "Champion", 7
    elif last["result"] == 1:
        res, stage = f"Won {last['round']}, did not play on", STAGE[last["round"]] + 1
    else:
        res = {"F": "Runner-up"}.get(last["round"], "Lost in " + last["round"])
        stage = STAGE[last["round"]]
    yr_rows.append({"player": PLAYER, "file_year": int(yr), "matches": len(g), "wins": int(g["result"].sum()),
                    "result": res, "stage": stage})
years_tbl = pd.DataFrame(yr_rows)
print(years_tbl.drop(columns=["player"]).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Wins above expected = sum of (result - expected probability) from our Elo v2, per 100 matches; "
               "hard-court events among the 14 analysed events, 2007-2025; retirements count; bars: 95% bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    events_tbl[k_] = v_
    years_tbl[k_] = v_
events_tbl.to_sql("player_hard_events", con, if_exists="replace", index=False)
years_tbl.to_sql("player_paris_by_year", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(events_tbl), "event rows,", len(years_tbl), "yearly rows")

In [ ]:
FIRST, LAST = 2007, 2025
py_tbl = pd.read_sql("SELECT file_year, result FROM player_paris_by_year", con)
paris_years = sorted(py_tbl["file_year"].tolist())
champ = set(py_tbl.loc[py_tbl["result"] == "Champion", "file_year"])

mm = pd.read_sql("""SELECT match_id, file_year, event_no, winner_id FROM matches
                    WHERE (winner_id = :p OR loser_id = :p) AND file_year BETWEEN :a AND :b""",
                 con, params={"p": pid, "a": FIRST, "b": LAST})
mm = mm[mm["file_year"].isin(paris_years)].copy()
mm["won"] = (mm["winner_id"] == pid).astype(float)
mm["is_paris"] = mm["event_no"].astype(str) == "352"
mm["one"] = 1.0
ids = set(mm["match_id"])

sv = pd.read_sql("SELECT * FROM server_rows WHERE file_year BETWEEN :a AND :b", con, params={"a": FIRST, "b": LAST})
sv = sv[sv["match_id"].isin(ids)]
mine = sv[sv["server_id"] == pid].copy()
opp = sv[sv["server_id"] != pid].copy()

mine["sv_n"] = mine["first_won"] + mine["second_won"]
mine["sv_d"] = mine["svpt"]
mine["ace_n"] = mine["ace"]
hold_num = mine["sv_games"] - (mine["bp_faced"] - mine["bp_saved"])
okh = hold_num >= 0
mine["hold_n"] = np.where(okh, hold_num, 0)
mine["hold_d"] = np.where(okh, mine["sv_games"], 0)

opp["ret_n"] = opp["svpt"] - (opp["first_won"] + opp["second_won"])
opp["ret_d"] = opp["svpt"]
okb = opp["bp_saved"] <= opp["bp_faced"]
opp["bpc_n"] = np.where(okb, opp["bp_faced"] - opp["bp_saved"], 0)
opp["bpc_d"] = np.where(okb, opp["bp_faced"], 0)

pmx = (mm.merge(mine[["match_id", "sv_n", "sv_d", "ace_n", "hold_n", "hold_d"]], on="match_id", how="left")
         .merge(opp[["match_id", "ret_n", "ret_d", "bpc_n", "bpc_d"]], on="match_id", how="left").fillna(0))

M = {
    "win_pct": ("won", "one", 100),
    "service_pts_won_pct": ("sv_n", "sv_d", 100),
    "return_pts_won_pct": ("ret_n", "ret_d", 100),
    "aces_pct": ("ace_n", "sv_d", 100),
    "service_games_held_pct": ("hold_n", "hold_d", 100),
    "break_points_converted_pct": ("bpc_n", "bpc_d", 100),
}

rows = []
for yr in paris_years:
    for label, mask in [("Paris Masters", pmx["is_paris"]), ("Rest of his season", ~pmx["is_paris"])]:
        g = pmx[(pmx["file_year"] == yr) & mask]
        for metric, (nc, dc, sc) in M.items():
            den = g[dc].sum()
            if den > 0:
                rows.append({"player": PLAYER, "metric": metric, "group": label, "file_year": int(yr),
                             "value": sc * g[nc].sum() / den, "matches": len(g), "champion": int(yr in champ)})
season_tbl = pd.DataFrame(rows)

rng = np.random.default_rng(31)


def boot_ratio_one(g, nc, dc, sc, B=500):
    n, d = g[nc].to_numpy(float), g[dc].to_numpy(float)
    idx = rng.integers(0, len(n), size=(B, len(n)))
    return sc * n[idx].sum(axis=1) / d[idx].sum(axis=1)


pp, rr = pmx[pmx["is_paris"]], pmx[~pmx["is_paris"]]
pooled_rows = []
for metric, (nc, dc, sc) in M.items():
    pv, rv = sc * pp[nc].sum() / pp[dc].sum(), sc * rr[nc].sum() / rr[dc].sum()
    lo, hi = np.percentile(boot_ratio_one(pp, nc, dc, sc) - boot_ratio_one(rr, nc, dc, sc), [2.5, 97.5])
    pooled_rows.append({"player": PLAYER, "metric": metric, "paris": pv, "rest_of_season": rv, "difference": pv - rv,
                        "ci_low": float(lo), "ci_high": float(hi),
                        "paris_matches": len(pp), "rest_matches": len(rr), "years": f"{min(paris_years)}-{max(paris_years)}"})
season_pooled = pd.DataFrame(pooled_rows)
print(len(pp), "Paris matches in", len(paris_years), "years |", len(rr), "other matches those years")
print(season_pooled[["metric", "paris", "rest_of_season", "difference", "ci_low", "ci_high"]].round(1).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Per year he played the Paris Masters: Paris vs the rest of his tour-level season in our data (Olympics, Davis Cup "
               "and team events not included). Retired matches excluded from serve and return statistics; service games held "
               "and break points exclude rows with impossible counts. Pooled difference: 95% bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    season_tbl[k_] = v_
    season_pooled[k_] = v_
season_tbl.to_sql("player_season_vs_paris", con, if_exists="replace", index=False)
season_pooled.to_sql("player_season_vs_paris_pooled", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(season_tbl), "yearly rows,", len(season_pooled), "pooled rows")

In [ ]:
import re
import numpy as np

PLAYER = "Novak Djokovic"
pid = pd.read_sql("SELECT player_id FROM players WHERE name = :n", con, params={"n": PLAYER})["player_id"].iloc[0]

mm = pd.read_sql("""SELECT file_year, tourney_name, event_no, best_of, retired, score, winner_id
                    FROM matches WHERE winner_id = :p OR loser_id = :p""", con, params={"p": pid})
mm["event_no"] = mm["event_no"].astype(str)
mm["won"] = (mm["winner_id"] == pid).astype(int)
SET = re.compile(r"(\d+)-(\d+)(?:\((\d+)\))?")
mm["n_sets"] = mm["score"].astype(str).apply(lambda s: len(SET.findall(s)))
mm["bracket"] = mm["score"].astype(str).str.contains(r"\[", regex=True)
mm["usable"] = (mm["retired"] == 0) & (~mm["bracket"])

FMT3 = "Best-of-3: went to a third set"
FMT5 = "Best-of-5: straight sets (3-0 or 0-3)"
cols = ["file_year", "event_no", "tourney_name", "won", "format", "origin"]

three = mm[mm["usable"] & (mm["n_sets"] == 3)].copy()
three["format"] = np.where(three["best_of"] == 3, FMT3, FMT5)
three["origin"] = "TML data"
three_all = three[cols]

man_path = root / "data" / "own" / "manual_results.csv"
if man_path.exists():
    man = pd.read_csv(man_path, dtype={"event_no": str})
    man = man[(man["player"] == PLAYER) & (man["n_sets"] == 3) & man["result"].isin(["W", "L"])].copy()
    man["file_year"] = pd.to_datetime(man["date"]).dt.year
    in_tml = ((mm["event_no"] == "352") & (mm["file_year"] == 2026)).any()
    if in_tml:
        skip = (man["event_no"] == "352") & (man["file_year"] == 2026)
        print("TML now has Paris 2026: ignoring", int(skip.sum()), "manual row(s). Delete them from manual_results.csv.")
        man = man[~skip]
    if len(man):
        man["won"] = (man["result"] == "W").astype(int)
        man["format"] = np.where(man["best_of"] == 3, FMT3, FMT5)
        man["origin"] = "added by hand"
        man["tourney_name"] = man["tournament"]
        three_all = pd.concat([three_all, man[cols]], ignore_index=True)
        print(len(man), "manual result(s) added.", "Still marked doubtful:",
              int((man["doubtful"].astype(str).str.lower() == "yes").sum()))


def tab(g, by):
    t = g.groupby(by).agg(matches=("won", "size"), wins=("won", "sum"))
    t["losses"] = t["matches"] - t["wins"]
    return t.astype(int)


g26 = three_all[three_all["file_year"] == 2026]
m26 = mm[mm["file_year"] == 2026]
print(f"2026: {len(m26)} matches in the TML data, {int((~m26['usable']).sum())} excluded (retired or bracketed tiebreak)")
print(f"2026, exactly three sets: {int(g26['won'].sum())} wins, {int(len(g26) - g26['won'].sum())} losses")
print(tab(g26, ["format", "origin"]).to_string())

pthree = three_all[three_all["event_no"] == "352"]
par = mm[mm["event_no"] == "352"]
print(f"Paris, all years: {int(pthree['won'].sum())} wins, {int(len(pthree) - pthree['won'].sum())} losses "
      f"in {len(pthree)} three-set matches (of {len(par)} Paris matches in the TML data)")
print(tab(pthree, ["file_year", "format", "origin"]).to_string())

out = pd.concat([
    tab(g26, ["format", "origin"]).reset_index().assign(group="2026", file_year=2026),
    tab(pthree, ["file_year", "format", "origin"]).reset_index().assign(group="Paris Masters"),
], ignore_index=True)
out.insert(0, "player", PLAYER)
meta = {
    "source": "Calculated from TennisMyLife match data; rows marked 'added by hand' come from data/own/manual_results.csv",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Completed matches that lasted exactly three sets (retired matches and bracketed match tiebreaks excluded; "
               "walkovers are not in the data). In best-of-5 matches this means a straight-sets result. "
               "Results not yet in the TML files are added by hand and marked."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    out[k_] = v_
out.to_sql("player_three_set_matches", con, if_exists="replace", index=False)
con.commit()
print("saved", len(out), "rows")

In [ ]:
import re

SETS = re.compile(r"(\d+)-(\d+)(?:\((\d+)\))?")
ROUND_ORDER = {"R128": 1, "R64": 2, "R32": 3, "R16": 4, "RR": 5, "QF": 6, "SF": 7, "F": 8}

s26 = pd.read_sql("""SELECT event_no, tourney_name, tourney_start, date, round, round_order, best_of, retired, score, winner_id
                     FROM matches WHERE (winner_id = :p OR loser_id = :p) AND file_year = 2026""",
                  con, params={"p": pid})
s26["event_no"] = s26["event_no"].astype(str)
s26["n_sets"] = s26["score"].astype(str).apply(lambda s: len(SETS.findall(s)))
s26["bracket"] = s26["score"].astype(str).str.contains(r"\[", regex=True)
dec = s26[(s26["retired"] == 0) & (~s26["bracket"]) & (s26["n_sets"] == 3) & (s26["best_of"] == 3)].copy()
dec["result"] = np.where(dec["winner_id"] == pid, "W", "L")
dec["origin"] = "TML data"
keep = ["tourney_name", "tourney_start", "date", "round", "round_order", "result", "origin"]
dec = dec[keep]

man_path = root / "data" / "own" / "manual_results.csv"
if man_path.exists():
    m = pd.read_csv(man_path, dtype={"event_no": str})
    m = m[(m["player"] == PLAYER) & (m["n_sets"] == 3) & (m["best_of"] == 3) & m["result"].isin(["W", "L"])].copy()
    m["dt"] = pd.to_datetime(m["date"])
    m = m[m["dt"].dt.year == 2026]
    if (s26["event_no"] == "352").any():
        skip = m["event_no"] == "352"
        print("TML now has Paris 2026: ignoring", int(skip.sum()), "manual row(s). Delete them from manual_results.csv.")
        m = m[~skip]
    if len(m):
        m["tourney_start"] = (m["dt"] - pd.to_timedelta(m["dt"].dt.weekday, unit="D")).dt.strftime("%Y-%m-%d")
        m["date"] = m["dt"].dt.strftime("%Y-%m-%d")
        m["round_order"] = m["round"].map(ROUND_ORDER).fillna(9)
        m["origin"] = "added by hand"
        m = m.rename(columns={"tournament": "tourney_name"})
        dec = pd.concat([dec, m[keep]], ignore_index=True)

dec = dec.sort_values(["date", "round_order"]).reset_index(drop=True)
dec["seq"] = dec.index + 1
print(dec[["seq", "date", "tourney_start", "tourney_name", "round", "result", "origin"]].to_string(index=False))


def longest(res, ch):
    best = cur = 0
    for r in res:
        cur = cur + 1 if r == ch else 0
        best = max(best, cur)
    return best


print("record:", int((dec["result"] == "W").sum()), "-", int((dec["result"] == "L").sum()),
      "| longest winning run:", longest(dec["result"], "W"), "| longest losing run:", longest(dec["result"], "L"))

meta = {
    "source": "Calculated from TennisMyLife match data; rows marked 'added by hand' come from data/own/manual_results.csv",
    "licence": "Derived from TML data; non-commercial, see data/Sources.md. Match-level: not for publication as a table.",
    "method": ("Best-of-3 matches that went to a third set in 2026, completed matches only, in order of date and round. "
               "Dates for TML rows are the date column of the 2026 file; hand-added rows use the real match date."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    dec[k_] = v_
dec.to_sql("player_third_set_sequence_2026", con, if_exists="replace", index=False)
con.commit()
print("saved", len(dec), "rows")

In [ ]:
import numpy as np

PLAYER = "Carlos Alcaraz"
LAST = 2025
STAGE = {"R128": 0, "R64": 1, "R32": 2, "R16": 3, "QF": 4, "SF": 5, "F": 6}

try:
    pd.read_sql("SELECT 1 FROM server_rows LIMIT 1", con)
except Exception:
    raise RuntimeError("Run the Paris analysis cell that creates the server_rows view first.")

ids_ = pd.read_sql("SELECT player_id, name FROM players WHERE name LIKE :n", con, params={"n": "%" + PLAYER.split()[-1] + "%"})
print(ids_.to_string(index=False))
pid = ids_.loc[ids_["name"] == PLAYER, "player_id"].iloc[0]

mm = pd.read_sql("""
    SELECT m.match_id, m.file_year, m.event_no, m.round, m.round_order, m.surface, e.result AS won, e.expected
    FROM matches m JOIN elo_long e ON e.match_id = m.match_id AND e.player_id = :p
    WHERE m.file_year <= :l
""", con, params={"p": pid, "l": LAST})
mm["event_no"] = mm["event_no"].astype(str)
mm["is_paris"] = mm["event_no"] == "352"
paris_years = sorted(int(y) for y in mm.loc[mm["is_paris"], "file_year"].unique())
assert paris_years, "No Paris matches found for this player"
champ = set(mm[(mm["is_paris"]) & (mm["round"] == "F") & (mm["won"] == 1)]["file_year"])
mm = mm[mm["file_year"].isin(paris_years)].copy()
mm["one"] = 1.0
mm["wae"] = mm["won"] - mm["expected"]

res_rows = []
for yr in paris_years:
    g = mm[mm["is_paris"] & (mm["file_year"] == yr)].sort_values("round_order")
    last = g.iloc[-1]
    if last["round"] == "F" and last["won"] == 1:
        stage, text = 7, "Champion"
    elif last["won"] == 1:
        stage, text = STAGE.get(last["round"], 0) + 1, "Won " + last["round"]
    else:
        stage, text = STAGE.get(last["round"], 0), ("Runner-up" if last["round"] == "F" else "Lost in " + last["round"])
    res_rows.append({"player": PLAYER, "file_year": yr, "matches": len(g), "wins": int(g["won"].sum()),
                     "result": text, "stage": stage})
results_tbl = pd.DataFrame(res_rows)

sv = pd.read_sql("SELECT * FROM server_rows WHERE file_year <= :l", con, params={"l": LAST})
sv = sv[sv["match_id"].isin(set(mm["match_id"]))]
mine = sv[sv["server_id"] == pid].copy()
opp = sv[sv["server_id"] != pid].copy()
mine["sv_n"] = mine["first_won"] + mine["second_won"]
mine["sv_d"] = mine["svpt"]
mine["ace_n"] = mine["ace"]
mine["fin_n"] = mine["first_in"]
mine["df_n"] = mine["df"]
hold_num = mine["sv_games"] - (mine["bp_faced"] - mine["bp_saved"])
okh = hold_num >= 0
mine["hold_n"] = np.where(okh, hold_num, 0)
mine["hold_d"] = np.where(okh, mine["sv_games"], 0)
opp["ret_n"] = opp["svpt"] - (opp["first_won"] + opp["second_won"])
opp["ret_d"] = opp["svpt"]
okb = opp["bp_saved"] <= opp["bp_faced"]
opp["bpc_n"] = np.where(okb, opp["bp_faced"] - opp["bp_saved"], 0)
opp["bpc_d"] = np.where(okb, opp["bp_faced"], 0)

pmx = (mm.merge(mine[["match_id", "sv_n", "sv_d", "ace_n", "fin_n", "df_n", "hold_n", "hold_d"]], on="match_id", how="left")
         .merge(opp[["match_id", "ret_n", "ret_d", "bpc_n", "bpc_d"]], on="match_id", how="left").fillna(0))

M = {
    "wins_above_expected_per_100": ("wae", "one", 100),
    "win_pct": ("won", "one", 100),
    "service_pts_won_pct": ("sv_n", "sv_d", 100),
    "first_serve_in_pct": ("fin_n", "sv_d", 100),
    "return_pts_won_pct": ("ret_n", "ret_d", 100),
    "aces_pct": ("ace_n", "sv_d", 100),
    "double_faults_pct": ("df_n", "sv_d", 100),
    "service_games_held_pct": ("hold_n", "hold_d", 100),
    "break_points_converted_pct": ("bpc_n", "bpc_d", 100),
}
GROUPS = {
    "Paris Masters": pmx["is_paris"],
    "Rest of his season": ~pmx["is_paris"],
    "Other hard-court matches": (~pmx["is_paris"]) & (pmx["surface"] == "Hard"),
}

rows = []
for yr in paris_years:
    for label, mask in GROUPS.items():
        g = pmx[(pmx["file_year"] == yr) & mask]
        for metric, (nc, dc, sc) in M.items():
            den = g[dc].sum()
            if den > 0:
                rows.append({"player": PLAYER, "metric": metric, "group": label, "file_year": yr,
                             "value": sc * g[nc].sum() / den, "matches": len(g), "champion": int(yr in champ)})
by_year_tbl = pd.DataFrame(rows)

rng = np.random.default_rng(41)


def boot_vals(g, nc, dc, sc, B=500):
    n, d = g[nc].to_numpy(float), g[dc].to_numpy(float)
    idx = rng.integers(0, len(n), size=(B, len(n)))
    den = d[idx].sum(axis=1)
    return np.where(den > 0, sc * n[idx].sum(axis=1) / np.where(den > 0, den, 1), np.nan)


pp = pmx[pmx["is_paris"]]
pooled_rows = []
for ref in ["Rest of his season", "Other hard-court matches"]:
    rr = pmx[GROUPS[ref]]
    for metric, (nc, dc, sc) in M.items():
        if pp[dc].sum() == 0 or rr[dc].sum() == 0:
            continue
        pv, rv = sc * pp[nc].sum() / pp[dc].sum(), sc * rr[nc].sum() / rr[dc].sum()
        lo, hi = np.nanpercentile(boot_vals(pp, nc, dc, sc) - boot_vals(rr, nc, dc, sc), [2.5, 97.5])
        pooled_rows.append({"player": PLAYER, "metric": metric, "reference": ref, "paris": pv, "reference_value": rv,
                            "difference": pv - rv, "ci_low": float(lo), "ci_high": float(hi),
                            "paris_matches": len(pp), "reference_matches": len(rr),
                            "years": f"{min(paris_years)}-{max(paris_years)}"})
pooled_tbl = pd.DataFrame(pooled_rows)

print(len(pp), "Paris matches in", len(paris_years), "years:", paris_years)
print(results_tbl.drop(columns=["player"]).to_string(index=False))
print(pooled_tbl[pooled_tbl["reference"] == "Other hard-court matches"][
    ["metric", "paris", "reference_value", "difference", "ci_low", "ci_high", "reference_matches"]].round(1).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Years in which the player has Paris Masters matches (through %d) vs his other matches in those years. "
               "Wins above expected = sum of (result - expected probability) from our Elo v2, per 100 matches; retirements count. "
               "Serve and return statistics exclude retired matches; service games held and break points exclude rows with impossible counts. "
               "Unforced errors are not in the TML files. Pooled difference: 95%% bootstrap over matches." % LAST),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}


def save_player(name, new):
    for k_, v_ in meta.items():
        new[k_] = v_
    try:
        old = pd.read_sql(f"SELECT * FROM {name}", con)
        new = pd.concat([old[old["player"] != PLAYER], new], ignore_index=True)
    except Exception:
        pass
    new.to_sql(name, con, if_exists="replace", index=False)


save_player("player_compare_by_year", by_year_tbl)
save_player("player_compare_pooled", pooled_tbl)
save_player("player_compare_results", results_tbl)
con.commit()
print("saved for", PLAYER)

In [ ]:
import numpy as np

LAST = 2025
PAR_B, PAR_D = "Paris (Bercy, 2015-2024)", "Paris (La Défense, 2025)"

sr = pd.read_sql("SELECT * FROM server_rows WHERE file_year BETWEEN 2015 AND :l", con, params={"l": LAST})
sr["spw_num"] = sr["first_won"] + sr["second_won"]
hold_num = sr["sv_games"] - (sr["bp_faced"] - sr["bp_saved"])
okh = hold_num >= 0
sr["hold_ok"] = np.where(okh, hold_num, 0)
sr["games_ok"] = np.where(okh, sr["sv_games"], 0)
cols_sum = ["spw_num", "ace", "df", "first_in", "svpt", "hold_ok", "games_ok"]
sums = sr.groupby("match_id")[cols_sum].sum().reset_index()

base = pd.read_sql("""
    SELECT m.match_id, m.file_year, m.event_no, m.tourney_name, m.surface, m.indoor, m.best_of, m.minutes,
           m.score, m.retired, e.expected AS p_winner
    FROM matches m JOIN elo_long e ON e.match_id = m.match_id AND e.result = 1
    WHERE m.file_year BETWEEN 2015 AND :l
""", con, params={"l": LAST})
allm = base.merge(sums, on="match_id", how="left")
allm[cols_sum] = allm[cols_sum].fillna(0)
allm["event_no"] = allm["event_no"].astype(str)

p = allm["p_winner"]
allm["fav_den"] = (p != 0.5).astype(float)
allm["p_fav"] = np.maximum(p, 1 - p)
allm["fav_won"] = (p > 0.5).astype(float)
allm["fav_gap"] = (allm["fav_won"] - allm["p_fav"]) * allm["fav_den"]
allm["band3"] = (allm["p_fav"] > 0.75).astype(float) * allm["fav_den"]
bo3 = (allm["best_of"] == 3) & (allm["retired"] == 0)
allm["min_den"] = (bo3 & allm["minutes"].notna()).astype(float)
allm["min_num"] = np.where(allm["min_den"] == 1, allm["minutes"], 0.0)
allm["tb_den"] = bo3.astype(float)
allm["tb_num"] = (bo3 & allm["score"].astype(str).str.contains("7-6|6-7", regex=True)).astype(float)

name_map = allm[["event_no", "file_year", "tourney_name"]]
latest_name = name_map.sort_values("file_year").groupby("event_no")["tourney_name"].last()


def assign_units(df):
    u = df["event_no"].astype(str).map(latest_name).to_numpy(dtype=object)
    par = (df["event_no"].astype(str) == "352").to_numpy()
    y = df["file_year"].to_numpy()
    u[par & (y <= 2024)] = PAR_B
    u[par & (y >= 2025)] = PAR_D
    return u


pe = allm[(allm["indoor"] == "I") & (allm["surface"] == "Hard")].copy()
pe["unit"] = assign_units(pe)
counts = pe.groupby("unit").size()
keep = [u for u, n in counts.items() if n >= 100 or str(u).startswith("Paris")]
pe = pe[pe["unit"].isin(keep)]
print(len(keep), "indoor hard-court units;", len(pe), "matches")
print(counts.loc[keep].sort_values(ascending=False).to_string())

METRICS = {
    "service_pts_won_pct": ("spw_num", "svpt", 100),
    "aces_pct": ("ace", "svpt", 100),
    "double_faults_pct": ("df", "svpt", 100),
    "first_serve_in_pct": ("first_in", "svpt", 100),
    "service_games_held_pct": ("hold_ok", "games_ok", 100),
    "minutes_per_match_bo3": ("min_num", "min_den", 1),
    "matches_with_tiebreak_pct_bo3": ("tb_num", "tb_den", 100),
    "favourite_win_pct": ("fav_won", "fav_den", 100),
    "favourite_wins_above_expected_pct": ("fav_gap", "fav_den", 100),
    "matches_with_75_plus_favourite_pct": ("band3", "fav_den", 100),
}
rng = np.random.default_rng(51)
B = 300


def boot_vals(g, nc, dc, sc):
    n, d = g[nc].to_numpy(float), g[dc].to_numpy(float)
    idx = rng.integers(0, len(n), size=(B, len(n)))
    den = d[idx].sum(axis=1)
    return np.where(den > 0, sc * n[idx].sum(axis=1) / np.where(den > 0, den, 1), np.nan)


rows = []
for u, g in pe.groupby("unit"):
    for metric, (nc, dc, sc) in METRICS.items():
        den = g[dc].sum()
        if den <= 0:
            continue
        lo, hi = np.nanpercentile(boot_vals(g, nc, dc, sc), [2.5, 97.5])
        rows.append({"metric": metric, "unit": u, "matches": len(g), "value": sc * g[nc].sum() / den,
                     "ci_low": float(lo), "ci_high": float(hi), "is_paris": int(str(u).startswith("Paris"))})
prof = pd.DataFrame(rows)
prof["rank"] = prof.groupby("metric")["value"].rank(ascending=False, method="min").astype(int)
prof["n_units"] = prof.groupby("metric")["unit"].transform("count")

peers = pe[~pe["unit"].astype(str).str.startswith("Paris")]
drows = []
for u in [PAR_B, PAR_D]:
    g = pe[pe["unit"] == u]
    if g.empty:
        continue
    for metric, (nc, dc, sc) in METRICS.items():
        if g[dc].sum() <= 0 or peers[dc].sum() <= 0:
            continue
        pv, rv = sc * g[nc].sum() / g[dc].sum(), sc * peers[nc].sum() / peers[dc].sum()
        lo, hi = np.nanpercentile(boot_vals(g, nc, dc, sc) - boot_vals(peers, nc, dc, sc), [2.5, 97.5])
        drows.append({"metric": metric, "unit": u, "paris": pv, "peers": rv, "difference": pv - rv,
                      "ci_low": float(lo), "ci_high": float(hi), "paris_matches": len(g), "peer_matches": len(peers),
                      "unit_type": "Bercy" if u == PAR_B else "La Défense",
                      "unit_unit": "minutes" if metric == "minutes_per_match_bo3" else "percentage points"})
diff = pd.DataFrame(drows)

print(diff[diff["unit"] == PAR_B][["metric", "paris", "peers", "difference", "ci_low", "ci_high"]].round(1).to_string(index=False))
print(prof[prof["unit"] == PAR_B][["metric", "value", "rank", "n_units"]].round(1).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("All tour-level indoor hard-court events in the TML data, 2015-2025, with Paris split by venue. Units with fewer than 100 "
               "matches are left out (Paris is kept). Serve statistics exclude retired matches; service games held excludes rows with "
               "impossible counts; minutes and tiebreaks use best-of-3 matches only. Favourite = the player rated higher by our Elo v2; "
               "retirements count. Intervals: 95% bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    prof[k_] = v_
    diff[k_] = v_
prof.to_sql("indoor_hard_event_profile", con, if_exists="replace", index=False)
diff.to_sql("paris_vs_indoor_hard_peers", con, if_exists="replace", index=False)
con.commit()
print("saved", len(prof), "profile rows,", len(diff), "difference rows")

In [ ]:
for name in ["pd", "np", "con", "LAST", "PAR_B", "assign_units", "boot_vals", "pe"]:
    print(name, name in globals())
print(pd.read_sql("""SELECT name, type FROM sqlite_master WHERE name IN
    ('serve_obs', 'server_rows', 'indoor_hard_event_profile', 'paris_vs_indoor_hard_peers', 'serve_advantage_indoor_hard')""",
                  con).to_string(index=False))

In [ ]:
import numpy as np

missing = [n for n in ["LAST", "PAR_B", "PAR_D", "assign_units", "boot_vals"] if n not in globals()]
if missing:
    raise RuntimeError(f"Run Cell 1 of the indoor-hard profile first (missing: {missing}).")

obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface, indoor,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)


def window_sums(df, keys, prefix):
    base_ = df.groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    parts = [base_.assign(file_year=base_["file_year"] + d) for d in (-1, 0, 1)]
    out = pd.concat(parts).groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    return out.rename(columns={"svpt": prefix + "_svpt", "val": prefix + "_val"})


def role_excess(t, d, pid_col, sf, prefix, mu, k):
    dd = d.assign(pid=d[pid_col])
    tt = t.assign(pid=t[pid_col])
    w = window_sums(dd, sf + ["pid"], prefix + "w")
    e = window_sums(dd, sf + ["pid", "event_no"], prefix + "e")
    tt = tt.merge(w, on=sf + ["pid", "file_year"], how="left")
    tt = tt.merge(e, on=sf + ["pid", "event_no", "file_year"], how="left")
    n = (tt[prefix + "w_svpt"].fillna(0) - tt[prefix + "e_svpt"].fillna(0)).to_numpy()
    v = (tt[prefix + "w_val"].fillna(0) - tt[prefix + "e_val"].fillna(0)).to_numpy()
    return (v - mu * n) / (n + k), n


def expected_values_k(obs, value, keys, k=500, analysed_only=True):
    base_cols = ["match_id", "file_year", "event_no", "analysed", "server_id", "returner_id", "svpt"]
    d = obs.rename(columns={value: "val"})[list(dict.fromkeys(base_cols + keys + ["val"]))]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)
    t = t.merge(window_sums(d, keys, "wt"), on=keys + ["file_year"], how="left")
    t = t.merge(window_sums(d, keys + ["event_no"], "et"), on=keys + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()
    ex_s, n_s = role_excess(t, d, "server_id", keys, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", keys, "r", mu, k)
    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t


res_all = expected_values_k(obs, "sv_won", ["surface", "indoor"], analysed_only=False)
res_all = res_all[(res_all["file_year"] <= LAST) & (res_all["indoor"] == "I") & (res_all["surface"] == "Hard")].copy()
res_all["unit"] = assign_units(res_all)
pm_ = res_all.groupby(["match_id", "unit"], as_index=False).agg(rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
pm_ = pm_[pm_["rows"] == 2]
cnt = pm_.groupby("unit").size()

rows = []
for u in [u for u, n in cnt.items() if n >= 100 or str(u).startswith("Paris")]:
    g = pm_[pm_["unit"] == u]
    if len(g) < 20:
        continue
    lo, hi = np.nanpercentile(boot_vals(g, "resid", "svpt", 100), [2.5, 97.5])
    rows.append({"unit": u, "matches": len(g), "serve_excess_pp": 100 * g["resid"].sum() / g["svpt"].sum(),
                 "ci_low": float(lo), "ci_high": float(hi), "is_paris": int(str(u).startswith("Paris"))})
adj = pd.DataFrame(rows).sort_values("serve_excess_pp", ascending=False)
print(adj.round(2).to_string(index=False))

meta2 = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Service points won above expectation (percentage points). Baseline: other events with the same surface and roof, same "
               "players, +/-1 year, shrinkage k=500; the event itself is left out. Indoor hard-court events, 2015-2025, Paris split by "
               "venue. Interval: 95% bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta2.items():
    adj[k_] = v_
adj.to_sql("serve_advantage_indoor_hard", con, if_exists="replace", index=False)
con.commit()
print("saved", len(adj), "rows")

In [ ]:
import numpy as np

missing = [n for n in ["res_all", "boot_vals", "pe", "LAST"] if n not in globals()]
if missing:
    raise RuntimeError(f"Run the indoor-hard cells first (missing: {missing}).")

par = res_all[res_all["event_no"].astype(str) == "352"]
pmy = par.groupby(["match_id", "file_year"], as_index=False).agg(rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
pmy = pmy[pmy["rows"] == 2]
rows = []
for yr, g in pmy.groupby("file_year"):
    if len(g) < 20:
        continue
    lo, hi = np.nanpercentile(boot_vals(g, "resid", "svpt", 100), [2.5, 97.5])
    rows.append({"file_year": int(yr), "matches": len(g), "serve_excess_pp": 100 * g["resid"].sum() / g["svpt"].sum(),
                 "ci_low": float(lo), "ci_high": float(hi), "provisional": 0})
adj_y = pd.DataFrame(rows)

pp = pe[pe["event_no"] == "352"]
aces_y = pp.groupby("file_year").agg(ace=("ace", "sum"), svpt=("svpt", "sum")).reset_index()
aces_y["aces_pct"] = 100 * aces_y["ace"] / aces_y["svpt"]
par_year = adj_y.merge(aces_y[["file_year", "aces_pct"]], on="file_year", how="left")
print(par_year.round(2).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Per Paris edition: aces as a share of service points (retired matches excluded) and service points won above "
               "expectation (percentage points; baseline: other indoor hard-court events, same players, +/-1 year, shrinkage k=500; "
               "interval: 95% bootstrap over matches). Provisional rows come from the in-progress file and may change."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
out = par_year.copy()
for k_, v_ in meta.items():
    out[k_] = v_
out.to_sql("paris_court_speed_outcomes", con, if_exists="replace", index=False)

cpi = pd.read_csv(root / "data" / "own" / "paris_cpi.csv")
cpi["licence"] = "Own table: values reported by third-party sources (see source); not TML data"
cpi.to_sql("paris_cpi", con, if_exists="replace", index=False)
con.commit()
print("saved", len(out), "outcome rows and", len(cpi), "CPI rows")

In [ ]:
print(pd.read_sql("SELECT file_year, matches, provisional FROM paris_court_speed_outcomes", con).to_string(index=False))
print(pd.read_sql("SELECT year, cpi, primary_ FROM (SELECT year, cpi, \"primary\" AS primary_ FROM paris_cpi)", con).head(3))

In [ ]:
import numpy as np

missing = [n for n in ["par_year", "meta", "obs", "expected_values_k", "boot_vals"] if n not in globals()]
if missing:
    raise RuntimeError(f"Rerun Cell A (and the cell that defines expected_values_k) first; missing: {missing}")

live_path = root / "data" / "raw" / "ongoing" / "ongoing_tourneys_2026-11-05.csv"
o = pd.read_csv(live_path)
o = o[o["tourney_id"].astype(str).str.endswith("-352")].copy()
o = o[~o["score"].astype(str).str.contains("RET|W/O|DEF", regex=True)].reset_index(drop=True)
o["match_id"] = 9_000_000 + o.index
print(len(o), "completed Paris 2026 matches;", o["round"].value_counts().to_dict())


def side(df, s):
    w = s == "w"
    return pd.DataFrame({
        "match_id": df["match_id"], "file_year": 2026, "event_no": "352", "analysed": 1, "surface": "Hard", "indoor": "I",
        "server_id": df["winner_id"] if w else df["loser_id"],
        "returner_id": df["loser_id"] if w else df["winner_id"],
        "svpt": df[f"{s}_svpt"], "sv_won": df[f"{s}_1stWon"] + df[f"{s}_2ndWon"], "ace": df[f"{s}_ace"]})


live = pd.concat([side(o, "w"), side(o, "l")], ignore_index=True).dropna()
live = live[live["svpt"] > 0].copy()
for c in ["svpt", "sv_won", "ace"]:
    live[c] = live[c].astype(int)
live["server_id"] = live["server_id"].astype(str)
live["returner_id"] = live["returner_id"].astype(str)
print("service rows used:", len(live))

print("ID types: database", obs["server_id"].dtype, "| live", live["server_id"].dtype)
print("sample database IDs:", list(obs["server_id"].astype(str).head(3)), "| sample live IDs:", list(live["server_id"].head(3)))
known = live["server_id"].isin(set(obs["server_id"].astype(str))).mean()
print(f"share of live servers whose ID is in the database: {100 * known:.0f}%")
if known < 0.8:
    raise RuntimeError("Most live player IDs are not in the database; the ID formats differ. Paste the lines above and stop here.")

obs2 = obs.copy()
obs2["server_id"] = obs2["server_id"].astype(str)
obs2["returner_id"] = obs2["returner_id"].astype(str)
obs_plus = pd.concat([obs2, live], ignore_index=True)

res_live = expected_values_k(obs_plus, "sv_won", ["surface", "indoor"], analysed_only=False)
rl = res_live[(res_live["file_year"] == 2026) & (res_live["event_no"].astype(str) == "352")]
m = rl.groupby("match_id", as_index=False).agg(rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
m = m[m["rows"] == 2]
lo, hi = np.nanpercentile(boot_vals(m, "resid", "svpt", 100), [2.5, 97.5])
row26 = {"file_year": 2026, "matches": len(m), "serve_excess_pp": 100 * m["resid"].sum() / m["svpt"].sum(),
         "ci_low": float(lo), "ci_high": float(hi), "provisional": 1,
         "aces_pct": 100 * live["ace"].sum() / live["svpt"].sum()}

par_year = pd.concat([par_year[par_year["file_year"] != 2026], pd.DataFrame([row26])], ignore_index=True)
out = par_year.copy()
for k_, v_ in meta.items():
    out[k_] = v_
out.to_sql("paris_court_speed_outcomes", con, if_exists="replace", index=False)
con.commit()

print(pd.read_sql("SELECT file_year, matches, provisional, ROUND(serve_excess_pp, 2) AS serve_pp, ROUND(aces_pct, 2) AS aces_pct "
                  "FROM paris_court_speed_outcomes ORDER BY file_year", con).to_string(index=False))

In [ ]:
from pathlib import Path
import sqlite3
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent
con = sqlite3.connect(root / "data" / "db" / "tennis.db")
(root / "charts").mkdir(exist_ok=True)
tables = {}


def load(table):
    try:
        return pd.read_sql(f"SELECT * FROM {table}", con)
    except Exception as e:
        print("could not load", table, "->", e)
        return None


PAL = {"paris": "#E4002B", "tour": "#8D99AE", "indoor": "#0077B6", "accent": "#FF9F1C", "ink": "#1D3557", "win": "#12B76A"}
CRED = "Data source: TennisMyLife — https://stats.tennismylife.org."
mpl.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.grid": True,
                     "grid.color": "#E5E7EB", "grid.linewidth": 1, "font.size": 13, "axes.titleweight": "bold"})


def finish(fig, headline, footer, filename, subtitle=None):
    wrapped = textwrap.fill(footer, 200)
    h = fig.get_size_inches()[1]
    bottom = (0.19 * (wrapped.count("\n") + 1) + 0.45) / h
    top = 1 - 0.95 / h
    fig.tight_layout(rect=(0.02, bottom, 0.98, top))
    fig.text(0.02, 1 - 0.12 / h, headline, fontsize=21, fontweight="bold", ha="left", va="top")
    if subtitle:
        fig.text(0.02, 1 - 0.5 / h, subtitle, fontsize=13, color="#555555", ha="left", va="top")
    fig.text(0.02, 0.2 / h, wrapped, fontsize=9.5, color="#555555", ha="left", va="bottom")
    fig.savefig(root / "charts" / filename, dpi=150, facecolor="white")
    plt.show()
    print("saved", root / "charts" / filename)

In [ ]:
import json
import urllib.request
from datetime import date

path = root / "data" / "own" / "venues.csv"
v = pd.read_csv(path)
print(len(v), "rows read from venues.csv")

places = v[["venue", "city", "latitude", "longitude"]].dropna(subset=["venue"]).drop_duplicates().reset_index(drop=True)
missing = places[places["latitude"].isna() | places["longitude"].isna()]
if len(missing):
    print("Fill latitude and longitude for:")
    print(missing[["venue", "city"]].to_string(index=False))
    raise RuntimeError("Coordinates missing")
print(len(places), "distinct venues to look up")


def elevation(lat, lon, show=False):
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.load(r)
    if show:
        print("raw response from the API:", data)
    val = data["elevation"]
    val = val[0] if isinstance(val, list) else val
    if val is None:
        raise ValueError(f"No elevation returned for {lat}, {lon}")
    return float(val)


alts = []
for i, r in enumerate(places.itertuples()):
    try:
        alts.append(elevation(r.latitude, r.longitude, show=(i == 0)))
    except Exception as e:
        print("FAILED for", r.venue, "->", e)
        raise
places["altitude_m"] = alts
print(places.sort_values("altitude_m", ascending=False)[["venue", "city", "altitude_m"]].to_string(index=False))

v2 = v.merge(places[["venue", "city", "altitude_m"]], on=["venue", "city"], how="left")
v2["altitude_source"] = "Open-Meteo elevation API (terrain model at the given coordinates), consulted " + str(date.today())
v2["licence"] = "Own table; coordinates from the cited source; elevation by Open-Meteo.com (CC BY 4.0, non-commercial use)"
v2.to_sql("event_venues", con, if_exists="replace", index=False)
con.commit()
print("saved", len(v2), "rows to event_venues")

In [ ]:
print(pd.read_sql("SELECT * FROM serve_advantage_pooled", con).to_string(index=False))

In [ ]:
t = pd.read_sql("SELECT name, surface, indoor_outdoor FROM tournaments", con)
print(t.to_string(index=False))

pooled = pd.read_sql("SELECT name, matches, serve_excess_pp, ci_low, ci_high FROM serve_advantage_pooled", con)
ev = pd.read_sql("SELECT tournament, year_from, year_to, years, venue, altitude_m FROM event_venues", con)


def editions(r):
    ys = range(max(int(r.year_from), 2015), min(int(r.year_to), 2025) + 1)
    if r.years == "odd":
        ys = [y for y in ys if y % 2 == 1]
    elif r.years == "even":
        ys = [y for y in ys if y % 2 == 0]
    return len(list(ys))


ev["editions"] = ev.apply(editions, axis=1)
ev = ev[ev["editions"] > 0].copy()
ev["alt_x_ed"] = ev["altitude_m"] * ev["editions"]
g = ev.groupby("tournament").agg(alt_x_ed=("alt_x_ed", "sum"), editions=("editions", "sum"),
                                 altitude_min_m=("altitude_m", "min"), altitude_max_m=("altitude_m", "max"),
                                 venues=("venue", "nunique")).reset_index()
g["altitude_mean_m"] = g["alt_x_ed"] / g["editions"]
g = g.drop(columns="alt_x_ed")

d = pooled.merge(g, left_on="name", right_on="tournament", how="left").drop(columns="tournament")
d = d.merge(t, on="name", how="left")
bad = d[d[["altitude_mean_m", "surface"]].isna().any(axis=1)]
if len(bad):
    print(bad[["name", "altitude_mean_m", "surface"]])
    raise RuntimeError("Some events did not match; the names differ between tables.")

d["source"] = "Calculated from TennisMyLife match data; venue table is our own"
d["licence"] = "Derived aggregate; non-commercial, see data/Sources.md; elevation by Open-Meteo.com (CC BY 4.0)"
d["method"] = ("Pooled 2015-2025 serve advantage per event (see serve_advantage_pooled) joined to venue altitude. Altitude is the "
               "average over the editions played at each venue (elevation model at the venue coordinates, roughly +/-20 m); "
               "min and max shown where the event moved or alternates between venues.")
d["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
d.to_sql("altitude_vs_serve", con, if_exists="replace", index=False)
con.commit()
print(d[["name", "altitude_mean_m", "altitude_min_m", "altitude_max_m", "venues", "surface", "indoor_outdoor", "serve_excess_pp"]]
      .round(1).sort_values("altitude_mean_m", ascending=False).to_string(index=False))

In [ ]:
by = pd.read_sql("SELECT * FROM serve_advantage_by_year", con)
print("columns:", list(by.columns))
if "half_width" not in by.columns and {"ci_low", "ci_high"} <= set(by.columns):
    by["half_width"] = (by["ci_high"] - by["ci_low"]) / 2
need = {"name", "file_year", "matches", "serve_excess_pp", "half_width"}
if need - set(by.columns):
    raise RuntimeError(f"serve_advantage_by_year is missing columns: {need - set(by.columns)}")
by = by[["name", "file_year", "matches", "serve_excess_pp", "half_width"]]

ev = pd.read_sql("SELECT tournament, year_from, year_to, years, venue, city, altitude_m FROM event_venues", con)
rows = []
for r in ev.itertuples():
    for y in range(int(r.year_from), int(r.year_to) + 1):
        if r.years == "odd" and y % 2 == 0:
            continue
        if r.years == "even" and y % 2 == 1:
            continue
        rows.append((r.tournament, y, r.venue, r.city, r.altitude_m))
ev_y = pd.DataFrame(rows, columns=["name", "file_year", "venue", "city", "altitude_m"])
if ev_y.duplicated(["name", "file_year"]).any():
    print(ev_y[ev_y.duplicated(["name", "file_year"], keep=False)])
    raise RuntimeError("Two venues for the same event and year: check the years column in venues.csv")

t = pd.read_sql("SELECT name, surface, indoor_outdoor FROM tournaments", con)
d = by.merge(ev_y, on=["name", "file_year"], how="left").merge(t, on="name", how="left")
bad = d[d[["venue", "altitude_m", "surface"]].isna().any(axis=1)]
if len(bad):
    print(bad[["name", "file_year"]].to_string(index=False))
    raise RuntimeError("Some event-years have no venue; check venues.csv year ranges.")

d["significant"] = (d["serve_excess_pp"].abs() > d["half_width"]).astype(int)
d["source"] = "Calculated from TennisMyLife match data; venue table is our own"
d["licence"] = "Derived aggregate; non-commercial, see data/Sources.md; elevation by Open-Meteo.com (CC BY 4.0)"
d["method"] = ("Serve advantage per event and year (see serve_advantage_by_year) joined to the venue that hosted that edition. "
               "significant = 1 when the 95% interval excludes zero.")
d["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
d.to_sql("altitude_by_year", con, if_exists="replace", index=False)
con.commit()

print("events missing from the by-year table:", sorted(set(t["name"]) - set(d["name"])))
print(d.pivot(index="name", columns="file_year", values="serve_excess_pp").round(1).to_string())
print("saved", len(d), "rows")

In [ ]:
q = """
SELECT t.name, t.surface, t.indoor_outdoor,
       COUNT(DISTINCT o.match_id) AS matches,
       ROUND(100.0 * SUM(o.sv_won) / SUM(o.svpt), 1) AS service_pts_won_pct,
       ROUND(100.0 * SUM(o.ace) / SUM(o.svpt), 1) AS aces_pct
FROM serve_obs o
JOIN tournaments t ON o.event_no = t.event_no
WHERE o.analysed = 1 AND o.file_year BETWEEN 2015 AND 2025
GROUP BY t.name
ORDER BY service_pts_won_pct DESC
"""
print(pd.read_sql(q, con).to_string(index=False))

In [ ]:
q = """
SELECT tourney_name, tourney_level, MIN(file_year) AS first_year, MAX(file_year) AS last_year, COUNT(*) AS matches
FROM matches
WHERE surface = 'Clay' AND file_year >= 2015
GROUP BY tourney_name, tourney_level
HAVING COUNT(*) >= 100
ORDER BY matches DESC
"""
print(pd.read_sql(q, con).to_string(index=False))

In [ ]:
print(pd.read_sql("SELECT file_year, matches, serve_excess_pp, half_width FROM altitude_by_year WHERE name = 'Madrid Masters' ORDER BY file_year", con).to_string(index=False))

In [ ]:
# only masters and gs
q = """
SELECT t.name, t.surface, t.indoor_outdoor,
       COUNT(DISTINCT o.match_id) AS matches,
       100.0 * SUM(o.sv_won) / SUM(o.svpt) AS service_pts_won_pct,
       100.0 * SUM(o.ace) / SUM(o.svpt) AS aces_pct
FROM serve_obs o
JOIN tournaments t ON o.event_no = t.event_no
WHERE o.analysed = 1 AND o.file_year BETWEEN 2015 AND 2025
GROUP BY t.name, t.surface, t.indoor_outdoor
"""
raw = pd.read_sql(q, con)
alt = pd.read_sql("SELECT name, altitude_mean_m, altitude_min_m, altitude_max_m FROM altitude_vs_serve", con)
raw = raw.merge(alt, on="name", how="left")
if raw["altitude_mean_m"].isna().any():
    print(raw[raw["altitude_mean_m"].isna()][["name"]])
    raise RuntimeError("Some events have no altitude; check altitude_vs_serve.")

raw["source"] = "Calculated from TennisMyLife match data; venue table is our own"
raw["licence"] = "Derived aggregate; non-commercial, see data/Sources.md; elevation by Open-Meteo.com (CC BY 4.0)"
raw["method"] = ("Share of service points won by the server and aces as a share of service points, pooled over all analysed matches "
                 "2015-2025 per event (retired and incomplete matches excluded), joined to venue altitude. Not adjusted for who plays.")
raw["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
raw.to_sql("serve_raw_by_event", con, if_exists="replace", index=False)
con.commit()
print(raw[["name", "surface", "matches", "service_pts_won_pct", "aces_pct", "altitude_mean_m"]].round(1)
      .sort_values(["surface", "altitude_mean_m"], ascending=[True, False]).to_string(index=False))

In [ ]:
#all atp tournaments
import re
import json
import time
import urllib.parse
import urllib.request
import numpy as np

# ---- 1. serve totals per event name, surface and roof (2015-2025) ----
g = pd.read_sql("""
SELECT m.tourney_name AS tname, MIN(m.event_no) AS event_no, m.surface, m.indoor,
       MAX(CASE WHEN m.tourney_level IN ('G','M','F') THEN 1 ELSE 0 END) AS is_big_level,
       COUNT(DISTINCT o.match_id) AS matches,
       SUM(o.svpt) AS svpt, SUM(o.sv_won) AS sv_won, SUM(o.ace) AS ace,
       MIN(o.file_year) AS first_year, MAX(o.file_year) AS last_year
FROM serve_obs o JOIN matches m ON m.match_id = o.match_id
WHERE o.file_year BETWEEN 2015 AND 2025
GROUP BY m.tourney_name, m.surface, m.indoor
""", con)

big = pd.read_sql("""SELECT t.event_no, t.name AS big_name, a.altitude_mean_m
                     FROM tournaments t JOIN altitude_vs_serve a ON a.name = t.name""", con)
g = g.merge(big, on="event_no", how="left")
g["is_big"] = (g["is_big_level"] == 1) & g["big_name"].notna()
print(len(g), "event-name groups;", int(g["is_big"].sum()), "belong to the 14 big events")

# ---- 2. look up each other event's city ----
FIX = {}   # corrections go here, for example {"St. Petersburg": "Saint Petersburg|RU"}
DROP = re.compile(r"\b(masters|open|cup|atp|championships?|international|classic|challenge|trophy|grand prix|world tour|finals?)\b", re.I)


def query_for(name):
    if name in FIX:
        return FIX[name]
    s = re.sub(r"\s+", " ", DROP.sub(" ", name)).strip(" -/.")
    return s if s else name


def get_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)


def geocode(query):
    place, _, cc = query.partition("|")
    url = "https://geocoding-api.open-meteo.com/v1/search?" + urllib.parse.urlencode(
        {"name": place, "count": 10, "language": "en", "format": "json"})
    res = get_json(url).get("results") or []
    if cc:
        res = [x for x in res if str(x.get("country_code", "")).upper() == cc.upper()]
    return res


def elevation(lat, lon):
    val = get_json(f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}")["elevation"]
    return float(val[0] if isinstance(val, list) else val)


todo = sorted(set(query_for(n) for n in g.loc[~g["is_big"], "tname"]))
print(len(todo), "places to look up")
rows = []
for i, qy in enumerate(todo):
    try:
        res = geocode(qy)
    except Exception as e:
        print("ERROR", qy, "->", e)
        res = []
    if i == 0:
        print("first raw result:", res[:1])
    time.sleep(0.15)
    if not res:
        rows.append({"query": qy, "ok": False})
        continue
    r0 = res[0]
    amb = False
    if len(res) > 1 and res[1].get("country_code") != r0.get("country_code"):
        p0, p1 = r0.get("population") or 0, res[1].get("population") or 0
        amb = not (p0 >= 5 * max(p1, 1))
    el = r0.get("elevation")
    if el is None:
        el = elevation(r0["latitude"], r0["longitude"])
    rows.append({"query": qy, "ok": True, "matched": r0.get("name"), "admin1": r0.get("admin1"), "cc": r0.get("country_code"),
                 "lat": r0["latitude"], "lon": r0["longitude"], "elev": float(el), "doubtful": "yes" if amb else "no"})
pl = pd.DataFrame(rows)

review = pl.copy()
review["matches"] = review["query"].map(
    g.assign(query=g["tname"].map(query_for)).groupby("query")["matches"].sum())
print(review.sort_values(["ok", "doubtful"], ascending=[True, False])
      [["query", "matched", "admin1", "cc", "elev", "doubtful", "matches"]].to_string(index=False))
failed = pl[~pl["ok"]]["query"].tolist()
if failed:
    print("\nNOT FOUND (add them to FIX and rerun):", failed)

# ---- 3. merge, pool and save ----
cols = ["key", "tname", "surface", "indoor", "matches", "svpt", "sv_won", "ace", "altitude_m", "alt_src", "country",
        "doubtful", "is_big_event", "first_year", "last_year"]
bg = g[g["is_big"]].assign(key=lambda d: d["big_name"], altitude_m=lambda d: d["altitude_mean_m"],
                           alt_src="venue (own venue table)", country="", doubtful="no", is_big_event=1)
og = g[~g["is_big"]].assign(query=lambda d: d["tname"].map(query_for)).merge(pl[pl["ok"]], on="query", how="inner")
og = og.assign(key=lambda d: d["lat"].round(2).astype(str) + "|" + d["lon"].round(2).astype(str), altitude_m=lambda d: d["elev"],
               alt_src="city (Open-Meteo geocoding)", country=lambda d: d["cc"], is_big_event=0)
allg = pd.concat([bg[cols], og[cols]], ignore_index=True).sort_values("matches", ascending=False)

res = allg.groupby(["key", "surface", "indoor"], as_index=False).agg(
    name=("tname", "first"), matches=("matches", "sum"), svpt=("svpt", "sum"), sv_won=("sv_won", "sum"), ace=("ace", "sum"),
    altitude_m=("altitude_m", "first"), alt_src=("alt_src", "first"), country=("country", "first"),
    doubtful=("doubtful", lambda s: "yes" if (s == "yes").any() else "no"), is_big_event=("is_big_event", "max"),
    names_merged=("tname", lambda s: " / ".join(dict.fromkeys(s))), first_year=("first_year", "min"), last_year=("last_year", "max"))
res["name"] = np.where(res["is_big_event"] == 1, res["key"], res["name"])
res["service_pts_won_pct"] = 100 * res["sv_won"] / res["svpt"]
res["aces_pct"] = 100 * res["ace"] / res["svpt"]
res = res[res["matches"] >= 100].drop(columns=["svpt", "sv_won", "ace"])
res["source"] = "Calculated from TennisMyLife match data; altitude from our venue table (big events) or Open-Meteo geocoding (others)"
res["licence"] = "Derived aggregate; non-commercial, see data/Sources.md; elevation by Open-Meteo.com (CC BY 4.0)"
res["method"] = ("Share of service points won and aces per serve point, pooled over matches 2015-2025 per place, surface and roof "
                 "(retired and incomplete matches excluded; places with under 100 matches dropped). Not adjusted for who plays. "
                 "Altitude of 250/500 events is the city elevation, not the venue.")
res["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
res.to_sql("serve_raw_all_events", con, if_exists="replace", index=False)
con.commit()
print("\nsaved", len(res), "events;", int((res["doubtful"] == "yes").sum()), "flagged doubtful")
print(res.groupby(["surface", "indoor"]).size())

In [ ]:
import json
import time
import urllib.parse
import urllib.request
import numpy as np

ed = pd.read_sql("""
SELECT t.name, t.level, m.event_no, m.file_year, MIN(m.tourney_start) AS start,
       MAX(CASE WHEN m.round = 'R128' THEN 1 ELSE 0 END) AS has_r128
FROM matches m JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1 AND m.file_year BETWEEN 2015 AND 2026 AND t.indoor_outdoor = 'O'
GROUP BY t.name, t.level, m.event_no, m.file_year
""", con)
ed["days"] = np.where(ed["level"] == "G", 14, np.where(ed["has_r128"] == 1, 12, 7))
ed["start"] = pd.to_datetime(ed["start"])
ed["end"] = ed["start"] + pd.to_timedelta(ed["days"] - 1, unit="D")

ev = pd.read_sql("SELECT tournament AS name, year_from, year_to, years, venue, city, latitude, longitude FROM event_venues", con)
rows = []
for r in ev.itertuples():
    for y in range(int(r.year_from), int(r.year_to) + 1):
        if r.years == "odd" and y % 2 == 0:
            continue
        if r.years == "even" and y % 2 == 1:
            continue
        rows.append((r.name, y, r.venue, r.city, r.latitude, r.longitude))
evy = pd.DataFrame(rows, columns=["name", "file_year", "venue", "city", "latitude", "longitude"])
ed = ed.merge(evy, on=["name", "file_year"], how="left")
if ed["latitude"].isna().any():
    print(ed[ed["latitude"].isna()][["name", "file_year"]])
    raise RuntimeError("Some editions have no venue; check venues.csv.")

cutoff = pd.Timestamp.today().normalize() - pd.Timedelta(days=7)
late = ed[ed["end"] > cutoff]
if len(late):
    print("Skipped (too recent for the archive):", list(zip(late["name"], late["file_year"])))
ed = ed[ed["end"] <= cutoff].reset_index(drop=True)
print(len(ed), "editions to look up")


def get_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)


def edition_weather(lat, lon, start, end, show=False):
    qs = urllib.parse.urlencode({"latitude": lat, "longitude": lon, "start_date": start.strftime("%Y-%m-%d"),
                                 "end_date": end.strftime("%Y-%m-%d"), "timezone": "auto",
                                 "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"})
    h = get_json("https://archive-api.open-meteo.com/v1/archive?" + qs)["hourly"]
    if show:
        print("hourly keys:", list(h.keys()), "| first values:", {k: v[:2] for k, v in h.items()})
    w = pd.DataFrame(h)
    w["time"] = pd.to_datetime(w["time"])
    day = w[(w["time"].dt.hour >= 11) & (w["time"].dt.hour <= 19)]
    rain = w.groupby(w["time"].dt.date)["precipitation"].sum()
    return {"temp_c": day["temperature_2m"].mean(), "humidity_pct": day["relative_humidity_2m"].mean(),
            "wind_kmh": day["wind_speed_10m"].mean(), "rain_days": int((rain > 1.0).sum()), "n_days": int(len(rain))}


out, failed = [], []
for i, r in enumerate(ed.itertuples()):
    try:
        out.append(edition_weather(r.latitude, r.longitude, r.start, r.end, show=(i == 0)))
    except Exception as e:
        failed.append((r.name, r.file_year, str(e)[:80]))
        out.append({"temp_c": np.nan, "humidity_pct": np.nan, "wind_kmh": np.nan, "rain_days": np.nan, "n_days": np.nan})
    time.sleep(0.3)
if failed:
    print("FAILED:", failed)

w = pd.concat([ed.reset_index(drop=True), pd.DataFrame(out)], axis=1)
w["start_date"] = w["start"].dt.strftime("%Y-%m-%d")
w["end_date"] = w["end"].dt.strftime("%Y-%m-%d")
w = w.drop(columns=["start", "end", "level", "event_no", "has_r128", "latitude", "longitude"])
w["source"] = "Open-Meteo historical weather (open-meteo.com), consulted " + pd.Timestamp.today().strftime("%Y-%m-%d")
w["licence"] = "Weather data by Open-Meteo.com (CC BY 4.0, non-commercial use)"
w["method"] = ("Mean of hourly values between 11:00 and 19:00 local time over the edition window (14 days Slams, 12 days for large draws, "
               "else 7), at the venue coordinates; rain_days = days with more than 1 mm. Model data, not a station reading; "
               "roof closures not recorded.")
w.to_sql("edition_weather", con, if_exists="replace", index=False)
con.commit()
print("saved", len(w), "rows,", int(w["temp_c"].isna().sum()), "without data")
print(w.pivot(index="name", columns="file_year", values="temp_c").round(0).to_string())

In [ ]:
sv = pd.read_sql("SELECT name, file_year, matches, serve_excess_pp, half_width, surface FROM altitude_by_year", con)
wx = pd.read_sql("SELECT name, file_year, venue, temp_c, humidity_pct, wind_kmh, rain_days FROM edition_weather", con)
d = sv.merge(wx, on=["name", "file_year"], how="inner")
d = d[d["file_year"] <= 2025].copy()
for c in ["serve_excess_pp", "temp_c", "humidity_pct", "wind_kmh"]:
    d[c + "_dev"] = d[c] - d.groupby(["name", "venue"])[c].transform("mean")
d["editions_in_group"] = d.groupby(["name", "venue"])["file_year"].transform("count")
d["use"] = (d["editions_in_group"] >= 5).astype(int)
d["source"] = "Calculated from TennisMyLife match data and Open-Meteo weather"
d["licence"] = "Derived aggregate; non-commercial, see data/Sources.md; weather by Open-Meteo.com (CC BY 4.0)"
d["method"] = ("Per edition: serve advantage and weather, each as a difference from the average of that event at the same venue "
               "(2015-2025). Only groups with at least 5 editions are used.")
d["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
d.to_sql("weather_vs_serve", con, if_exists="replace", index=False)
con.commit()
print(len(d), "editions,", int(d["use"].sum()), "used")
print(d[d["use"] == 1][["serve_excess_pp_dev", "temp_c_dev", "humidity_pct_dev", "wind_kmh_dev"]].corr().round(2).iloc[0])